# Supply-Shock Data Pipeline — Starter Notebook

**Goal:** gather the key data from open sources, normalize it onto three join keys
(**country codes · dates · coordinates**), join it into one table, and use that table
to *compose a story* — trace an event (e.g. a chokepoint disruption) through to the
airfields it affects.

**Sources in this starter (the open-API core):**

| Source | What it gives | Access |
|---|---|---|
| OurAirports | 80k+ airfields with coordinates | public-domain CSV |
| JODI-Oil | production / stocks by country | CSV download |
| World Bank | economic exposure (GDP, energy) | `wbgapi` API |
| UN Comtrade | trade flows (who imports crude) | API (free tier) |
| EIA | prices, stocks, crude imports | REST API (free key) |

> This is deliberately the *reliable, no-scraping* set. GDELT/ACLED (events),
> IMF PortWatch (chokepoint transits), and OPEC MOMR (PDF tables) are added as
> **Extensions** at the bottom.

**Principle:** you don't scrape these — you use APIs and downloads. Keep API keys in
environment variables, respect rate limits, and mind licensing (e.g. ACLED is
non-commercial only).


## 0 · Setup

Install once (uncomment), then import. Put your API keys in a `.env` file or your shell
environment — never hardcode them.


In [1]:
# Dependencies already installed. Uncomment and re-run only if you need to reinstall.
# %pip install -q pandas requests wbgapi comtradeapicall pycountry python-dotenv pyarrow openpyxl xlrd

import os, io, time, requests
import pandas as pd
from pathlib import Path

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass

CACHE = Path("data_cache"); CACHE.mkdir(exist_ok=True)

# Keys (set these in your environment / .env):
EIA_API_KEY  = os.environ.get("EIA_API_KEY", "")
COMTRADE_KEY = os.environ.get("COMTRADE_KEY", "")

# ── Per-source cache TTLs (hours) ─────────────────────────────────────────────
# Set to match each source's actual publication cadence, not a universal default.
CACHE_TTL = {
    "gdelt_discovery":    6,      # updates every 15 min; we refresh every 6h
    "eia_bulk":          24,      # daily prices/stocks
    "portwatch":         24,      # daily transit counts
    "eia_stocks":       168,      # weekly (7 days)
    "jodi":             720,      # monthly (~30 days); also has ~2 month publication lag
    "gpr":              720,      # monthly
    "pink_prices":      720,      # monthly
    "comtrade":         720,      # monthly
    "jodi_stocks":      720,      # monthly (same source as jodi)
    "jodi_refinery":    720,      # monthly
    "portwatch_v2":      24,      # daily (new key includes extra chokepoints)
    "worldbank":       8760,      # annual (365 days)
    "airports":        2160,      # rarely changes; refresh every 90 days
}
DEFAULT_TTL = 24   # fallback for any source not listed above


def cached(name, fetch_fn, max_age_hours=None):
    """
    Fetch via fetch_fn() and cache to parquet.
    Uses per-source TTL from CACHE_TTL if max_age_hours is not explicitly passed.
    Warns if data is older than 2x its TTL.
    """
    ttl   = max_age_hours if max_age_hours is not None else CACHE_TTL.get(name, DEFAULT_TTL)
    fp    = CACHE / f"{name}.parquet"
    now   = time.time()

    if fp.exists():
        age_hours = (now - fp.stat().st_mtime) / 3600
        if age_hours < ttl:
            return pd.read_parquet(fp)
        if age_hours > ttl * 2:
            print(f"  [STALE] {name}: last fetched {age_hours:.0f}h ago "
                  f"(TTL {ttl}h) — data may be significantly out of date.")

    df = fetch_fn()
    try:
        df.to_parquet(fp)
    except Exception as e:
        print(f"  [cache] could not write {fp.name} ({e}); returning uncached data.")
    return df


def cache_status():
    """
    Print a freshness summary for all cached sources:
    when last fetched, age vs TTL, and latest date in the data (if a date column exists).
    """
    print("=" * 72)
    print("CACHE STATUS")
    print("  {:<22}  {:<12}  {:<8}  {:<8}  {}".format(
          "Source", "Last fetched", "Age", "TTL", "Latest data date"))
    print("  {}  {}  {}  {}  {}".format("-"*22, "-"*12, "-"*8, "-"*8, "-"*18))

    for name in sorted(CACHE_TTL.keys()):
        fp  = CACHE / f"{name}.parquet"
        ttl = CACHE_TTL[name]
        if not fp.exists():
            print("  {:<22}  {:<12}  {:<8}  {:<8}  {}".format(
                  name, "not cached", "-", str(ttl)+"h", "-"))
            continue

        age_h   = (time.time() - fp.stat().st_mtime) / 3600
        age_str = "{:.0f}h ago".format(age_h)
        stale   = " !" if age_h > ttl * 2 else ""

        latest_date = "-"
        try:
            df = pd.read_parquet(fp)
            for col in ["date", "seendate", "TIME_PERIOD"]:
                if col in df.columns:
                    val = pd.to_datetime(df[col], errors="coerce").max()
                    if pd.notna(val):
                        latest_date = str(val.date())
                        break
        except Exception:
            pass

        print("  {:<22}  {:<12}  {:<8}  {:<8}  {}{}".format(
              name,
              pd.Timestamp(fp.stat().st_mtime, unit="s").strftime("%Y-%m-%d"),
              age_str,
              str(ttl) + "h",
              latest_date,
              stale))

    print("  ! = older than 2x TTL")
    print("=" * 72)

cache_status()


CACHE STATUS
  Source                  Last fetched  Age       TTL       Latest data date
  ----------------------  ------------  --------  --------  ------------------
  airports                2026-07-21    25h ago   2160h     -
  comtrade                not cached    -         720h      -
  eia_bulk                2026-07-21    24h ago   24h       2026-07-13
  eia_stocks              not cached    -         168h      -
  gdelt_discovery         2026-07-22    3h ago    6h        2026-07-22
  gpr                     2026-07-21    25h ago   720h      2026-06-01
  jodi                    not cached    -         720h      -
  jodi_refinery           2026-07-22    1h ago    720h      2026-05-01
  jodi_stocks             2026-07-22    1h ago    720h      2026-05-01
  pink_prices             2026-07-21    25h ago   720h      2026-06-01
  portwatch               2026-07-21    25h ago   24h       2026-07-19
  portwatch_v2            2026-07-22    1h ago    24h       2026-07-19
  worldbank    

## 1 · OurAirports — the base map (coordinates)

Public-domain CSV, no key. This is the board every other layer is overlaid on.
We keep the busier airports and the fields we need to join and locate them.


In [2]:
AIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"

def load_airports():
    df = pd.read_csv(AIRPORTS_URL)
    keep = ["ident","type","name","iso_country","latitude_deg","longitude_deg","iata_code"]
    df = df[keep]
    # focus on airports that actually take jet traffic
    df = df[df["type"].isin(["large_airport","medium_airport"])]
    return df.reset_index(drop=True)

airports = cached("airports", load_airports)
print(f"{len(airports):,} airports")
airports.head()


5,276 airports


,ident,type,name,iso_country,latitude_deg,longitude_deg,iata_code
0,07FA,medium_airport,Ocean Reef Club Airport,US,25.325399,-80.274803,OCA
1,5A8,medium_airport,Aleknagik / New Airport,US,59.282600,-158.617996,WKK
2,AE-0221,medium_airport,Sas Al Nakheel Air Base,AE,24.441280,54.516960,None
3,AG-0001,medium_airport,Burton-Nibbs International Airport,AG,17.621194,-61.798347,BBQ
4,AGGH,large_airport,Honiara International Airport,SB,-9.428000,160.054993,HIR


## 2 · JODI-Oil — production & stocks by country (join key: country)

JODI publishes the **World Database** as a CSV. Download the latest from
<https://www.jodidata.org/oil/database/data-downloads.aspx> and point `JODI_CSV` at the
file (it may sit behind a click, so a local path is most reliable). Expected schema below.


In [3]:
# After downloading, set this to your local file:
JODI_CSV = CACHE / "jodi_oil_world.csv"     # <-- put the downloaded CSV here

# JODI columns: REF_AREA (ISO2 country), ENERGY_PRODUCT, FLOW_BREAKDOWN,
#               UNIT_MEASURE, TIME_PERIOD (YYYY-MM), OBS_VALUE, ASSESSMENT_CODE
#
# NOTE: JODI reports every (country, month) in FIVE units (CONVBBL, KBBL, KBD, KL,
# KTONS). We MUST pin one unit, or a later groupby will mix them and the numbers
# become meaningless. KBD = thousand barrels/day, the standard production metric.
JODI_UNIT = "KBD"

def load_jodi():
    if not JODI_CSV.exists():
        print("JODI file not found — download it and set JODI_CSV. Skipping for now.")
        return pd.DataFrame(columns=["REF_AREA","ENERGY_PRODUCT","FLOW_BREAKDOWN",
                                     "TIME_PERIOD","OBS_VALUE"])
    df = pd.read_csv(JODI_CSV)
    # crude oil production (CRUDEOIL / INDPROD) in a single unit is the headline supply series
    m = ((df["ENERGY_PRODUCT"]=="CRUDEOIL") &
         (df["FLOW_BREAKDOWN"]=="INDPROD") &
         (df["UNIT_MEASURE"]==JODI_UNIT))
    out = df[m].copy()
    out["OBS_VALUE"] = pd.to_numeric(out["OBS_VALUE"], errors="coerce")  # "-" placeholders -> NaN
    return out

jodi = load_jodi()
jodi.head()


def load_jodi_stocks():
    """
    Extract closing crude oil stocks from the JODI World Database.
    Searches for any flow code containing 'STK' or 'STOCK' (defensive — JODI
    codes vary slightly by version).  Converts to thousand barrels.
    Used to estimate strategic reserve buffer in the trace output.
    """
    if not JODI_CSV.exists():
        print("  JODI stocks: file not found — skipping")
        return pd.DataFrame()
    df = pd.read_csv(JODI_CSV)
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    avail = set(df["FLOW_BREAKDOWN"].unique())
    stk_flows = [f for f in avail if any(
        kw in str(f).upper() for kw in ["STK", "STOCK", "CLOSTK", "CLOSSTK"]
    )]
    if not stk_flows:
        print("  JODI stocks: no stock flow codes found in file")
        return pd.DataFrame()

    mask = (
        (df["ENERGY_PRODUCT"] == "CRUDEOIL") &
        (df["FLOW_BREAKDOWN"].isin(stk_flows))
    )
    stk = df[mask].copy()
    # Prefer KBBL; convert KTON if needed (1 tonne crude ≈ 7.33 barrels)
    kbbl = stk[stk["UNIT_MEASURE"] == "KBBL"].copy()
    kton = stk[stk["UNIT_MEASURE"] == "KTON"].copy()
    kton["OBS_VALUE"] = kton["OBS_VALUE"] * 7.33
    kton["UNIT_MEASURE"] = "KBBL"
    result = pd.concat([kbbl, kton], ignore_index=True)
    print("  JODI stocks: {:,} rows, {} countries, flows: {}".format(
          len(result), result["REF_AREA"].nunique() if not result.empty else 0,
          stk_flows[:4]))
    return result


def load_jodi_refinery():
    """
    Extract refinery crude throughput from the JODI World Database.
    Searches for flow codes containing 'REFPRO', 'REFIN', or 'CRDINP'.
    Used to show Singapore's refinery output (the regional jet fuel hub).
    """
    if not JODI_CSV.exists():
        return pd.DataFrame()
    df = pd.read_csv(JODI_CSV)
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    avail = set(df["FLOW_BREAKDOWN"].unique())
    ref_flows = [f for f in avail if any(
        kw in str(f).upper() for kw in ["REFPRO", "REFIN", "CRDINP", "CRDINT"]
    )]
    if not ref_flows:
        print("  JODI refinery: no refinery flow codes found in file")
        return pd.DataFrame()

    mask = (
        (df["ENERGY_PRODUCT"] == "CRUDEOIL") &
        (df["FLOW_BREAKDOWN"].isin(ref_flows)) &
        (df["UNIT_MEASURE"].isin(["KBD", "KBBL"]))
    )
    ref = df[mask].copy()
    print("  JODI refinery: {:,} rows, {} countries".format(
          len(ref), ref["REF_AREA"].nunique() if not ref.empty else 0))
    return ref


jodi_stocks   = cached("jodi_stocks",   load_jodi_stocks)
jodi_refinery = cached("jodi_refinery", load_jodi_refinery)


## 3 · World Bank — economic exposure (join key: country)

The `wbgapi` package needs no key. Pull indicators that tell you which economies are most
exposed to an oil-supply shock (GDP, oil rents, energy use).


In [4]:
import wbgapi as wb

INDICATORS = {
    "NY.GDP.MKTP.CD":  "gdp_usd",
    "NY.GDP.PETR.RT.ZS":"oil_rents_pct_gdp",
    "EG.USE.COMM.KT.OE":"energy_use_ktoe",
}

def load_worldbank():
    df = wb.data.DataFrame(list(INDICATORS), mrv=1)   # most-recent value
    df = df.rename(columns=INDICATORS).reset_index().rename(columns={"economy":"iso3"})
    return df

worldbank = cached("worldbank", load_worldbank)
worldbank.head()


,iso3,gdp_usd,oil_rents_pct_gdp
0,ABW,NaN,NaN
1,AFE,1.358694e+12,NaN
2,AFG,NaN,NaN
3,AFW,8.525270e+11,NaN
4,AGO,1.221749e+11,NaN


## 4 · UN Comtrade — who imports crude (join key: country)

Trade flows show the downstream exposure — which countries buy crude, and from whom.
Commodity code **2709** = crude petroleum oils. The free preview endpoint works without a
key (limited rows); a subscription key raises the limits.


In [5]:
import comtradeapicall
import datetime as _dt_ct

def _comtrade_period():
    """Most recent month Comtrade is likely to have published (2-3 month lag)."""
    y, m = _dt_ct.date.today().year, _dt_ct.date.today().month
    m -= 3
    if m <= 0:
        m += 12
        y -= 1
    return "{:04d}{:02d}".format(y, m)

def load_crude_imports(period=None):
    if period is None:
        period = _comtrade_period()
    print("  Comtrade: requesting period {} (HS 2709 = crude petroleum)...".format(period))
    try:
        df = comtradeapicall.previewFinalData(
            typeCode="C", freqCode="M", clCode="HS", period=period,
            reporterCode=None, cmdCode="2709", flowCode="M",
            partnerCode=None, partner2Code=None,
            customsCode=None, motCode=None, maxRecords=2500, format_output="JSON",
            countOnly=None, includeDesc=True,
        )
        if df is not None and not df.empty:
            print("  Comtrade: {:,} rows loaded for period {}".format(len(df), period))
        return df if df is not None else pd.DataFrame()
    except Exception as e:
        print("Comtrade preview failed (rate limit or network):", e)
        return pd.DataFrame()

# Cache key includes period so stale data is not silently reused when the month rolls over.
_ct_period = _comtrade_period()
crude_imports = cached("comtrade_" + _ct_period, lambda: load_crude_imports(_ct_period))
crude_imports.head()


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20260401,2026,4,202604,31,AZE,Azerbaijan,M,...,22345133.0,False,0.000,False,1.667737e+07,NaN,1.667737e+07,0,False,True
1,C,M,20260401,2026,4,202604,31,AZE,Azerbaijan,M,...,22345133.0,False,0.000,False,1.667737e+07,NaN,1.667737e+07,0,False,True
2,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,798227801.0,False,649044.111,False,6.760885e+08,6.437813e+08,6.760885e+08,0,False,True
3,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,145663031.0,False,116817.818,False,1.252029e+08,1.179866e+08,1.252029e+08,0,False,True
4,C,M,20260401,2026,4,202604,36,AUS,Australia,M,...,189670556.0,False,153880.865,False,1.514154e+08,1.438682e+08,1.514154e+08,0,False,True


## 5 · EIA — prices, stocks & crude imports (join key: date)

EIA offers the same data two ways: a keyed REST API, or **public bulk downloads that need
no key at all**. We use the **bulk** route — whole datasets ship as one zip of
newline-delimited JSON — so nothing here requires signing up. Attribute "U.S. EIA".

We pull a few headline series from the `PET` (Petroleum) dataset: WTI & Brent spot prices,
US crude imports, and crude stocks. Swap in other `series_id`s (e.g. from the `STEO`
forecast dataset) by editing `EIA_BULK`.


In [6]:
# EIA — NO API KEY NEEDED: use the public bulk downloads, not the keyed REST API.
# A whole dataset ships as one zip of newline-delimited JSON. Attribute "U.S. EIA".
import zipfile, json as _json

EIA_BULK = {   # EIA series_id -> friendly metric name   (all from the PET dataset)
    "PET.RWTC.D":                 "wti_usd_bbl",          # WTI spot price, daily
    "PET.RBRTE.D":                "brent_usd_bbl",        # Brent spot price, daily
    "PET.EER_EPJK_PF4_RGC_DPG.D": "jet_fuel_usd_gal",     # US Gulf Coast jet-fuel spot, daily (layer 4)
    "PET.MCRIMUS2.M":             "us_crude_imports_kbd", # US crude oil imports, monthly
    "PET.WCESTUS1.W":             "us_crude_stocks_kbbl", # US crude oil stocks, weekly (verify gate)
}

def _eia_bulk_zip(dataset="PET", max_age_hours=24):
    """Download an EIA bulk dataset zip (no key) into the cache, reusing if fresh."""
    fp = CACHE / f"{dataset}.zip"
    if not (fp.exists() and (time.time() - fp.stat().st_mtime) < max_age_hours*3600):
        r = requests.get(f"https://api.eia.gov/bulk/{dataset}.zip", timeout=180); r.raise_for_status()
        fp.write_bytes(r.content)
    return fp

def load_eia_bulk(series=EIA_BULK, dataset="PET"):
    fp = _eia_bulk_zip(dataset)
    want = set(series); rows = []
    with zipfile.ZipFile(fp) as z:
        inner = [n for n in z.namelist() if n.endswith(".txt")][0]
        with z.open(inner) as fh:
            for raw in io.TextIOWrapper(fh, encoding="utf-8"):
                if not want: break                              # stop once every series is found
                if not any(s in raw for s in want): continue    # cheap prefilter before json.loads
                o = _json.loads(raw); sid = o.get("series_id")
                if sid in want and o.get("data"):
                    rows += [(series[sid], p, v) for p, v in o["data"]]
                    want.discard(sid)
    df = pd.DataFrame(rows, columns=["metric", "period", "value"])
    p = df["period"].astype(str)                # periods are mixed: YYYYMMDD / YYYYMM / YYYY
    df["date"] = pd.NaT
    for L, fmt in [(8, "%Y%m%d"), (6, "%Y%m"), (4, "%Y")]:
        m = p.str.len() == L
        df.loc[m, "date"] = pd.to_datetime(p[m], format=fmt, errors="coerce")
    return df

eia = cached("eia_bulk", load_eia_bulk)
if not eia.empty:
    latest = (eia.sort_values("date").groupby("metric")
                 .agg(latest_date=("date","last"), latest_value=("value","last")))
    print("EIA (no key) — latest values:"); print(latest.to_string())
eia.head()


EIA (no key) — latest values:
                     latest_date  latest_value
metric                                        
brent_usd_bbl         2026-07-13        81.620
jet_fuel_usd_gal      2026-07-13         3.379
us_crude_imports_kbd  2026-04-01      6257.000
us_crude_stocks_kbbl  2026-07-10    409665.000
wti_usd_bbl           2026-07-13        79.200


,metric,period,value,date
0,wti_usd_bbl,20260713,79.20,2026-07-13
1,wti_usd_bbl,20260710,72.45,2026-07-10
2,wti_usd_bbl,20260709,73.15,2026-07-09
3,wti_usd_bbl,20260708,74.56,2026-07-08
4,wti_usd_bbl,20260707,71.53,2026-07-07


## 5b · Commercial-safe extension sources (no API key)

Four more sources that load with **no key** and are cleared for **commercial use** (just
attribute each). They fill the price, chokepoint-transit, geopolitical-risk, and news
layers of the story.

| Source | Layer it fills | License / attribution |
|---|---|---|
| World Bank Pink Sheet | crude & commodity **prices** | CC BY 4.0 — "World Bank Commodity Markets" |
| IMF PortWatch | **chokepoint** transit volumes | free w/ attribution — "IMF PortWatch" |
| GPR Index (Caldara–Iacoviello) | geopolitical-**risk** premium | free — cite Caldara & Iacoviello (2022) |
| GDELT 2.0 | global **news/event** signal | open — "The GDELT Project" |

> **ACLED is deliberately excluded.** Its free data is *non-commercial only*, so it can't
> go in a commercial product without a paid ACLED license. Everything above can.


In [7]:
# World Bank "Pink Sheet" — monthly commodity prices incl. crude (Brent/Dubai/WTI). CC BY 4.0.
# NOTE: the dated path segment (…-0050012026/…) changes when the workbook is refreshed.
# If this 404s, grab the current .xlsx link from worldbank.org/en/research/commodity-markets.
PINK_URL = ("https://thedocs.worldbank.org/en/doc/"
            "74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx")

def load_pink_prices():
    raw = pd.read_excel(PINK_URL, sheet_name="Monthly Prices", header=None)
    names = raw.iloc[4].tolist()               # row 4 holds the commodity names
    data = raw.iloc[6:].copy()                 # data rows start at row 6
    data.columns = ["period"] + list(names[1:])
    crude = [c for c in data.columns if isinstance(c, str) and c.startswith("Crude oil")]
    out = data[["period"] + crude].copy()
    out["date"] = pd.to_datetime(out["period"].astype(str).str.replace("M", "-", regex=False),
                                 format="%Y-%m", errors="coerce")
    for c in crude:
        out[c] = pd.to_numeric(out[c], errors="coerce")   # "…" placeholders -> NaN
    return out.dropna(subset=["date"]).drop(columns=["period"]).reset_index(drop=True)

pink_prices = cached("pink_prices", load_pink_prices)
if not pink_prices.empty:
    print(f"{len(pink_prices):,} months of prices, through {pink_prices['date'].max().date()}")
pink_prices.tail(3)


798 months of prices, through 2026-06-01


,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI",date
795,103.9,120.4,92.7,98.6,2026-04-01
796,100.4,107.5,94.7,99.1,2026-05-01
797,81.7,85.4,77.7,81.9,2026-06-01


In [8]:
# IMF PortWatch — daily transit counts & capacity at maritime chokepoints. Attribute "IMF PortWatch."
PW_CHOKE_URL = ("https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/"
                "Daily_Chokepoints_Data/FeatureServer/0/query")
# this notebook's short chokepoint names -> PortWatch "portname" values
PW_NAME = {
    "Hormuz":      "Strait of Hormuz",
    "Suez":        "Suez Canal",
    "Malacca":     "Malacca Strait",
    "BabelMandeb": "Bab el-Mandeb Strait",
    "Panama":      "Panama Canal",
    # Indo-Pacific alternative corridors
    "Lombok":      "Lombok Strait",
    "Sunda":       "Sunda Strait",
    "Taiwan":      "Taiwan Strait",
    "SCS":         "South China Sea",
}

def load_portwatch():
    names = "','".join(PW_NAME.values())
    params = {"where": f"portname IN ('{names}')",
              "outFields": "date,portname,n_tanker,n_cargo,n_total,capacity_tanker,capacity",
              "orderByFields": "date DESC",       # newest first; server caps at 1000 rows
              "resultRecordCount": 1000, "f": "json"}
    r = requests.get(PW_CHOKE_URL, params=params, timeout=40); r.raise_for_status()
    df = pd.DataFrame([f["attributes"] for f in r.json().get("features", [])])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

portwatch = cached("portwatch_v2", load_portwatch)
if not portwatch.empty:
    print(f"{len(portwatch):,} chokepoint-day rows, {portwatch['portname'].nunique()} chokepoints, "
          f"through {portwatch['date'].max().date()}")
portwatch.head()


1,000 chokepoint-day rows, 8 chokepoints, through 2026-07-19


,date,portname,n_tanker,n_cargo,n_total,capacity_tanker,capacity
0,2026-07-19,Suez Canal,21,22,43,1443001,1973255
1,2026-07-19,Panama Canal,18,16,34,449996,873923
2,2026-07-19,Bab el-Mandeb Strait,18,15,33,1031074,1341101
3,2026-07-19,Malacca Strait,65,105,170,3647281,7672211
4,2026-07-19,Strait of Hormuz,4,11,15,39216,182620


In [9]:
# GPR geopolitical-risk index (Caldara & Iacoviello, 2022). Free — cite the authors.
GPR_URL = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"

def load_gpr():
    g = pd.read_excel(GPR_URL, sheet_name="Sheet1").rename(columns={"month": "date"})
    keep = [c for c in ["date","GPR","GPRT","GPRA"] if c in g.columns]   # index, threats, acts
    g = g[keep].dropna(subset=["GPR"]).copy()
    g["date"] = pd.to_datetime(g["date"], errors="coerce")
    return g.dropna(subset=["date"]).reset_index(drop=True)

gpr = cached("gpr", load_gpr)
if not gpr.empty:
    print(f"{len(gpr):,} months; latest GPR {gpr['GPR'].iloc[-1]:.1f} ({gpr['date'].iloc[-1].date()})")
gpr.tail(3)


498 months; latest GPR 173.6 (2026-06-01)


,date,GPR,GPRT,GPRA
495,2026-04-01,254.429504,274.106720,310.083588
496,2026-05-01,203.556671,233.463852,215.219376
497,2026-06-01,173.639557,210.581039,177.959946


In [10]:
# GDELT 2.0 DOC API — broad energy-disruption discovery query. No key. Attribute "The GDELT Project."
# Two-window fetch: one call for the recent period, one for the prior baseline.
# This prevents the 250-article cap from filling entirely with recent articles,
# which would leave no prior baseline and make spike ratios meaningless.

GDELT_DISCOVERY_QUERY = (
    '(oil OR crude OR petroleum OR LNG OR tanker OR pipeline) '
    'AND (ceasefire OR sanctions OR attack OR conflict OR disruption '
    'OR blockade OR embargo OR escalation OR explosion OR shutdown)'
)

def load_gdelt_discovery(query=GDELT_DISCOVERY_QUERY, recent_days=30, prior_days=60, maxrecords=250):
    """
    Two separate GDELT API calls: last recent_days days (spike detection window)
    and the prior_days before that (baseline). Combining them gives detect_spikes()
    a real prior to compare against.
    """
    import datetime as _dt_g
    url   = "https://api.gdeltproject.org/api/v2/doc/doc"
    today = _dt_g.datetime.utcnow()

    def _fetch(start_dt, end_dt, label):
        params = {
            "query":         query,
            "mode":          "artlist",
            "format":        "json",
            "maxrecords":    maxrecords,
            "STARTDATETIME": start_dt.strftime("%Y%m%d%H%M%S"),
            "ENDDATETIME":   end_dt.strftime("%Y%m%d%H%M%S"),
            "sort":          "datedesc",
        }
        for attempt in range(4):
            try:
                r = requests.get(url, params=params, timeout=40)
                if r.status_code == 200:
                    arts = r.json().get("articles", [])
                    if arts:
                        df = pd.DataFrame(arts)
                        if "seendate" in df.columns:
                            df["seendate"] = pd.to_datetime(
                                df["seendate"], format="%Y%m%dT%H%M%SZ", errors="coerce"
                            )
                        print("  GDELT: {:,} articles ({})".format(len(df), label))
                        return df
            except Exception:
                pass
            time.sleep(2 * (attempt + 1))
        print("  GDELT: {} window failed after retries".format(label))
        return pd.DataFrame()

    r_start = today - _dt_g.timedelta(days=recent_days)
    recent  = _fetch(r_start, today, "last {} days".format(recent_days))

    p_end   = r_start
    p_start = today - _dt_g.timedelta(days=recent_days + prior_days)
    prior   = _fetch(p_start, p_end,
                     "{}-{} days ago".format(recent_days, recent_days + prior_days))

    combined = pd.concat([recent, prior], ignore_index=True)
    if not combined.empty and "url" in combined.columns:
        combined = combined.drop_duplicates(subset=["url"])
    print("  GDELT total: {:,} articles  ({} recent + {} prior baseline)".format(
          len(combined), len(recent), len(prior)))
    return combined

gdelt_discovery = cached("gdelt_discovery_2w", load_gdelt_discovery, max_age_hours=6)


## 6 · Normalize onto the join keys

Everything gets a consistent **ISO-3 country code**, parsed **dates**, and (for airports)
**coordinates**. `pycountry` maps names/ISO2 → ISO3.


In [11]:
import pycountry

def to_iso3(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if not x: return None
    try:
        if x.isdigit():                                    # Comtrade/M49 numeric code, e.g. "842" -> USA
            return pycountry.countries.get(numeric=x.zfill(3)).alpha_3
        if len(x)==2:  return pycountry.countries.get(alpha_2=x).alpha_3
        if len(x)==3:  return pycountry.countries.get(alpha_3=x).alpha_3
        return pycountry.countries.lookup(x).alpha_3       # fall back to a name lookup
    except Exception:
        return None                                        # aggregates like World ("000") land here

airports["iso3"] = airports["iso_country"].map(to_iso3)
if not jodi.empty:
    jodi["iso3"] = jodi["REF_AREA"].map(to_iso3)

if not crude_imports.empty:
    # reporter = the importing country; partner = who it bought the crude from
    rep = crude_imports["reporterISO"] if "reporterISO" in crude_imports else crude_imports.get("reporterCode")
    crude_imports["iso3"] = rep.map(to_iso3) if rep is not None else None
    par = crude_imports["partnerISO"] if "partnerISO" in crude_imports else crude_imports.get("partnerCode")
    crude_imports["partner_iso3"] = par.map(to_iso3) if par is not None else None
    print("comtrade rows with reporter iso3:", crude_imports["iso3"].notna().mean().round(2),
          "| with partner iso3:", crude_imports["partner_iso3"].notna().mean().round(2))

if "jodi_stocks" in globals() and not jodi_stocks.empty and "REF_AREA" in jodi_stocks.columns:
    jodi_stocks["iso3"] = jodi_stocks["REF_AREA"].map(to_iso3)
if "jodi_refinery" in globals() and not jodi_refinery.empty and "REF_AREA" in jodi_refinery.columns:
    jodi_refinery["iso3"] = jodi_refinery["REF_AREA"].map(to_iso3)

print("normalized. airports with iso3:", airports["iso3"].notna().mean().round(2))


comtrade rows with reporter iso3: 1.0 | with partner iso3: 0.79
normalized. airports with iso3: 1.0


## 7 · Assemble one country-level table

Merge the country-keyed sources (World Bank exposure, JODI production, Comtrade imports)
into a single frame. Airports stay in their own table, joined to countries by `iso3`
(and locatable by coordinates).


In [12]:
country = worldbank.copy() if not worldbank.empty else pd.DataFrame({"iso3":[]})

if not jodi.empty:
    prod = (jodi.sort_values("TIME_PERIOD").groupby("iso3")["OBS_VALUE"].last()
                .rename("crude_prod_latest").reset_index())
    country = country.merge(prod, on="iso3", how="outer")

if not crude_imports.empty and "iso3" in crude_imports:
    # Sum only bilateral rows (partner_iso3 present). Comtrade also returns a per-reporter
    # "World" total row whose partner won't map to an iso3 — including it would double-count.
    bilateral = crude_imports[crude_imports["partner_iso3"].notna()]
    imp = (bilateral.groupby("iso3")["primaryValue"].sum()
               .rename("crude_import_value").reset_index())
    country = country.merge(imp, on="iso3", how="outer")

country.to_parquet(CACHE / "country_assembled.parquet")
print(country.shape); country.head()


(266, 5)


,iso3,gdp_usd,oil_rents_pct_gdp,crude_prod_latest,crude_import_value
0,ABW,NaN,NaN,NaN,NaN
1,AFE,1.358694e+12,NaN,NaN,NaN
2,AFG,NaN,NaN,NaN,NaN
3,AFW,8.525270e+11,NaN,NaN,NaN
4,AGO,1.221749e+11,NaN,1084.0,NaN


In [13]:
## 8 · Graph Network — Node Definitions

# The oil supply chain for the Indo-Pacific is modelled as a directed graph.
# Five node types, each playing a different role in the network:
#
#   loading_terminal  — where crude is loaded onto tankers  (source)
#   chokepoint        — maritime corridors all routes pass through  (transit)
#   refinery          — where crude is converted to jet fuel  (processing)
#   storage           — strategic reserves / tank farms  (buffer)
#   airport           — airfields that consume jet fuel  (sink)
#
# Edges (routes between nodes) are built in the next section.
# Capacity values are approximate; sources: EIA, IEA, operator reports.

LOADING_TERMINALS = [
    {"node_id":"LT_RAS_TANURA",  "name":"Ras Tanura",          "iso3":"SAU","lat": 26.64,"lon": 50.16,"capacity_kbd":7000},
    {"node_id":"LT_KHARG",       "name":"Kharg Island",        "iso3":"IRN","lat": 29.26,"lon": 50.32,"capacity_kbd":5000},
    {"node_id":"LT_MINA_AHMADI", "name":"Mina Al-Ahmadi",      "iso3":"KWT","lat": 29.07,"lon": 48.13,"capacity_kbd":1700},
    {"node_id":"LT_FUJAIRAH",    "name":"Fujairah Terminal",   "iso3":"ARE","lat": 25.13,"lon": 56.34,"capacity_kbd":2000},
    {"node_id":"LT_RUWAIS",      "name":"Ruwais Terminal",     "iso3":"ARE","lat": 24.11,"lon": 52.73,"capacity_kbd": 800},
    {"node_id":"LT_BASRA",       "name":"Basra Oil Terminal",  "iso3":"IRQ","lat": 29.67,"lon": 48.78,"capacity_kbd":1800},
    {"node_id":"LT_SOHAR",       "name":"Sohar Terminal",      "iso3":"OMN","lat": 24.35,"lon": 56.62,"capacity_kbd": 800},
    {"node_id":"LT_RAS_LAFFAN",  "name":"Ras Laffan",          "iso3":"QAT","lat": 25.90,"lon": 51.55,"capacity_kbd":1200},
]

CHOKEPOINT_NODES = [
    {"node_id":"CP_HORMUZ",      "name":"Strait of Hormuz",     "iso3":None,"lat": 26.60,"lon": 56.50,"capacity_kbd":21000},
    {"node_id":"CP_MALACCA",     "name":"Malacca Strait",       "iso3":None,"lat":  2.50,"lon":101.40,"capacity_kbd":16800},
    {"node_id":"CP_LOMBOK",      "name":"Lombok Strait",        "iso3":None,"lat": -8.50,"lon":115.90,"capacity_kbd": 3500},
    {"node_id":"CP_SUNDA",       "name":"Sunda Strait",         "iso3":None,"lat": -6.00,"lon":105.50,"capacity_kbd": 2000},
    {"node_id":"CP_TAIWAN",      "name":"Taiwan Strait",        "iso3":None,"lat": 24.50,"lon":119.50,"capacity_kbd": 5500},
    {"node_id":"CP_SCS",         "name":"South China Sea",      "iso3":None,"lat": 14.00,"lon":114.00,"capacity_kbd":15000},
    {"node_id":"CP_BABELMANDEB", "name":"Bab el-Mandeb Strait", "iso3":None,"lat": 12.60,"lon": 43.30,"capacity_kbd": 6000},
    {"node_id":"CP_SUEZ",        "name":"Suez Canal",           "iso3":None,"lat": 30.00,"lon": 32.60,"capacity_kbd": 5500},
    {"node_id":"CP_INDIAN_OCEAN","name":"Indian Ocean Corridor","iso3":None,"lat": -5.00,"lon": 73.00,"capacity_kbd":20000},
]

REFINERY_NODES = [
    # Singapore — regional jet fuel hub for the Indo-Pacific
    {"node_id":"RF_SINGAPORE",   "name":"Singapore Jurong Island",    "iso3":"SGP","lat":  1.26,"lon":103.69,"capacity_kbd":1400},
    # South Korea
    {"node_id":"RF_ULSAN",       "name":"Ulsan Refinery Complex",     "iso3":"KOR","lat": 35.54,"lon":129.31,"capacity_kbd":1100},
    {"node_id":"RF_YEOSU",       "name":"Yeosu Refinery Complex",     "iso3":"KOR","lat": 34.74,"lon":127.73,"capacity_kbd": 800},
    # Japan
    {"node_id":"RF_CHIBA",       "name":"Chiba Refinery",             "iso3":"JPN","lat": 35.61,"lon":140.12,"capacity_kbd": 500},
    {"node_id":"RF_MIZUSHIMA",   "name":"Mizushima Refinery",         "iso3":"JPN","lat": 34.52,"lon":133.77,"capacity_kbd": 400},
    # Taiwan
    {"node_id":"RF_MAILIAO",     "name":"Mailiao Refinery (Formosa)", "iso3":"TWN","lat": 23.77,"lon":120.22,"capacity_kbd": 800},
    # India
    {"node_id":"RF_JAMNAGAR",    "name":"Jamnagar Refinery (Reliance)","iso3":"IND","lat": 22.47,"lon": 70.06,"capacity_kbd":1240},
    # Australia
    {"node_id":"RF_GEELONG",     "name":"Geelong Refinery (Viva)",    "iso3":"AUS","lat":-38.15,"lon":144.36,"capacity_kbd": 120},
    # Malaysia
    {"node_id":"RF_PORT_DICKSON","name":"Port Dickson Refinery",      "iso3":"MYS","lat":  2.52,"lon":101.80,"capacity_kbd": 156},
]

# spr_days = approximate days of net import cover the reserve provides
STORAGE_NODES = [
    {"node_id":"ST_JAPAN_SPR",  "name":"Japan Strategic Reserve",        "iso3":"JPN","lat": 36.20,"lon":138.25,"capacity_kbd":None,"spr_days":150},
    {"node_id":"ST_KOREA_SPR",  "name":"South Korea Strategic Reserve",  "iso3":"KOR","lat": 36.50,"lon":127.50,"capacity_kbd":None,"spr_days":100},
    {"node_id":"ST_SGP_COMM",   "name":"Singapore Commercial Storage",   "iso3":"SGP","lat":  1.26,"lon":103.69,"capacity_kbd":None,"spr_days": 80},
    {"node_id":"ST_TAIWAN_SPR", "name":"Taiwan Strategic Reserve",       "iso3":"TWN","lat": 23.70,"lon":120.50,"capacity_kbd":None,"spr_days": 60},
    {"node_id":"ST_AUS_SPR",    "name":"Australia Strategic Reserve",    "iso3":"AUS","lat":-25.00,"lon":133.00,"capacity_kbd":None,"spr_days": 54},
    {"node_id":"ST_INDIA_SPR",  "name":"India Strategic Reserve",        "iso3":"IND","lat": 20.00,"lon": 78.00,"capacity_kbd":None,"spr_days":  9},
    {"node_id":"ST_FUJAIRAH_HUB","name":"Fujairah Storage Hub",          "iso3":"ARE","lat": 25.13,"lon": 56.34,"capacity_kbd":None,"spr_days": None},
    {"node_id":"ST_CHINA_SPR",  "name":"China SPR (est.)",               "iso3":"CHN","lat": 29.90,"lon":122.10,"capacity_kbd":None,"spr_days": 90},
]

INDOPACIFIC_ISO3 = {
    "JPN","KOR","TWN","PHL","VNM","SGP","MYS","IDN",
    "AUS","NZL","THA","IND","CHN","BRN","PNG",
}

def build_network_nodes():
    frames = []

    for node_type, node_list in [
        ("loading_terminal", LOADING_TERMINALS),
        ("chokepoint",       CHOKEPOINT_NODES),
        ("refinery",         REFINERY_NODES),
        ("storage",          STORAGE_NODES),
    ]:
        df = pd.DataFrame(node_list)
        df["node_type"] = node_type
        frames.append(df)

    # Airports: pull large Indo-Pacific airports from the already-loaded airports table
    if "airports" in globals() and not airports.empty:
        ap = airports[
            airports["iso3"].isin(INDOPACIFIC_ISO3) &
            (airports["type"] == "large_airport")
        ][["ident","name","iso3","latitude_deg","longitude_deg"]].copy()
        ap = ap.rename(columns={"ident":"node_id","latitude_deg":"lat","longitude_deg":"lon"})
        ap["node_type"]    = "airport"
        ap["capacity_kbd"] = None
        ap["spr_days"]     = None
        frames.append(ap)

    nodes = pd.concat(frames, ignore_index=True)

    # Ensure consistent columns
    for col in ["capacity_kbd","spr_days"]:
        if col not in nodes.columns:
            nodes[col] = None

    return nodes

network_nodes = build_network_nodes()

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("GRAPH NETWORK  --  NODE INVENTORY")
print("=" * 60)
type_labels = {
    "loading_terminal": "Loading terminals  (crude source)",
    "chokepoint":       "Chokepoints        (transit corridors)",
    "refinery":         "Refineries         (processing nodes)",
    "storage":          "Storage / SPR      (buffer nodes)",
    "airport":          "Airports           (sink nodes)",
}
for ntype, label in type_labels.items():
    subset = network_nodes[network_nodes["node_type"] == ntype]
    if subset.empty:
        continue
    print("  {:<42}  {:>3} nodes".format(label, len(subset)))
    for _, r in subset.iterrows():
        cap = ""
        if pd.notna(r.get("capacity_kbd")):
            cap = "  {:,} kbd".format(int(r["capacity_kbd"]))
        spr = ""
        if pd.notna(r.get("spr_days")):
            spr = "  ~{} days cover".format(int(r["spr_days"]))
        iso = "  [{}]".format(r["iso3"]) if r.get("iso3") else ""
        print("    {}{}{}{}".format(r["name"], iso, cap, spr))
    print()
print("  {:<42}  {:>3} nodes total".format("", len(network_nodes)))
print("=" * 60)
print()
print("Next step: define the edges (routes) between these nodes.")
print("Use  network_nodes  DataFrame for downstream graph analysis.")


GRAPH NETWORK  --  NODE INVENTORY
  Loading terminals  (crude source)             8 nodes
    Ras Tanura  [SAU]  7,000 kbd
    Kharg Island  [IRN]  5,000 kbd
    Mina Al-Ahmadi  [KWT]  1,700 kbd
    Fujairah Terminal  [ARE]  2,000 kbd
    Ruwais Terminal  [ARE]  800 kbd
    Basra Oil Terminal  [IRQ]  1,800 kbd
    Sohar Terminal  [OMN]  800 kbd
    Ras Laffan  [QAT]  1,200 kbd

  Chokepoints        (transit corridors)        9 nodes
    Strait of Hormuz  21,000 kbd
    Malacca Strait  16,800 kbd
    Lombok Strait  3,500 kbd
    Sunda Strait  2,000 kbd
    Taiwan Strait  5,500 kbd
    South China Sea  15,000 kbd
    Bab el-Mandeb Strait  6,000 kbd
    Suez Canal  5,500 kbd
    Indian Ocean Corridor  20,000 kbd

  Refineries         (processing nodes)         9 nodes
    Singapore Jurong Island  [SGP]  1,400 kbd
    Ulsan Refinery Complex  [KOR]  1,100 kbd
    Yeosu Refinery Complex  [KOR]  800 kbd
    Chiba Refinery  [JPN]  500 kbd
    Mizushima Refinery  [JPN]  400 kbd
    Mailiao Refi

/var/folders/dg/b9h0v8_j2c32lbc607r_4ml00000gn/T/ipykernel_34464/2501766460.py:99: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  nodes = pd.concat(frames, ignore_index=True)


## 9 · Event Discovery

Three-gate pipeline that surfaces oil-relevant events from the news **without pre-specifying
geography or event type**. The data tells you what happened and where — you don't tell it
what to look for.

| Gate | Question | Method |
|---|---|---|
| **1 — News spike** | Is something being talked about significantly more? | Country mentions in GDELT titles: last 30d vs prior 60d, ≥5× ratio |
| **2 — Oil relevance** | Does this geography matter for oil supply? | JODI producer list (≥100 kbd) + chokepoint-adjacent countries |
| **3 — Market signal** | Did the market notice? | Brent/WTI or PortWatch tanker transit move >2% in same window |

**Output:** `INVESTIGATE` (all 3 gates) or `WATCH` (Gates 1–2, no market move yet).
The event's geography and direction (price up/down, transit up/down) come from the data.


In [14]:
import re

# ── Build name → ISO-3 lookup from pycountry ────────────────────────────────
_COUNTRY_NAME_MAP = {}
for _c in pycountry.countries:
    _COUNTRY_NAME_MAP[_c.name.lower()] = _c.alpha_3
    if hasattr(_c, "common_name"):
        _COUNTRY_NAME_MAP[_c.common_name.lower()] = _c.alpha_3
    if hasattr(_c, "official_name"):
        _COUNTRY_NAME_MAP[_c.official_name.lower()] = _c.alpha_3

# Aliases: short forms, adjectives, actor/group names, and chokepoint terms.
# Value = ISO-3 code, or None for multi-country concepts (skip those).
COUNTRY_ALIASES = {
    # English short forms & adjectives
    "us": "USA",  "u.s.": "USA",  "usa": "USA",  "american": "USA",
    "uk": "GBR",  "u.k.": "GBR",  "britain": "GBR",   "british": "GBR",
    "uae": "ARE", "emirati": "ARE", "emirates": "ARE",
    "iranian": "IRN",  "iraqi": "IRQ",   "saudi": "SAU",
    "russian": "RUS",  "chinese": "CHN", "venezuelan": "VEN",
    "nigerian": "NGA", "libyan": "LBY",  "kuwaiti": "KWT",
    "qatari": "QAT",   "omani": "OMN",   "bahraini": "BHR",
    "yemeni": "YEM",   "sudanese": "SDN","azerbaijani": "AZE",
    "kazakh": "KAZ",   "algerian": "DZA","angolan": "AGO",
    "egyptian": "EGY", "turkish": "TUR", "lebanese": "LBN",
    "syrian": "SYR",   "ukrainian": "UKR","indonesian": "IDN",
    "malaysian": "MYS","panamanian": "PAN","norwegian": "NOR",
    # Actor / group → country they operate from
    "houthi":    "YEM", "houthis":   "YEM",
    "hezbollah": "LBN",
    "hamas":     "PSE",
    "opec":      None,   # multi-country — skip
    # Chokepoint terms → the country that controls/sits on it
    "suez":          "EGY",
    "malacca":       "MYS",
    "hormuz":        "IRN",
    "bab el-mandeb": "YEM", "bab-el-mandeb": "YEM",
    "panama canal":  "PAN",
}


def extract_countries_from_title(title):
    """
    Return a list of ISO-3 codes for countries/regions mentioned in an article title.
    Checks COUNTRY_ALIASES first (longer / more specific phrases), then pycountry names.
    Skips names shorter than 4 chars to avoid false matches on common words.
    """
    t = str(title).lower()
    found = set()

    for alias, iso3 in COUNTRY_ALIASES.items():
        if iso3 and re.search(r"\b" + re.escape(alias) + r"\b", t):
            found.add(iso3)

    for name, iso3 in _COUNTRY_NAME_MAP.items():
        if len(name) >= 4 and re.search(r"\b" + re.escape(name) + r"\b", t):
            found.add(iso3)

    return list(found)


In [15]:
def detect_spikes(df, primary_days=30, threshold=5.0, min_recent=3):
    """
    Gate 1: find countries mentioned ≥ threshold× more often in the last primary_days
    than in the prior context window (normalised to the same length).

    min_recent guards against tiny-count noise (e.g. 0 → 1 article counting as infinite spike).
    Returns a DataFrame: iso3, recent_count, prior_count, prior_norm, spike_ratio.
    """
    if df.empty or "seendate" not in df.columns or "title" not in df.columns:
        return pd.DataFrame()

    df = df.copy()
    df["countries"] = df["title"].apply(extract_countries_from_title)

    # one row per country mention per article
    exploded = (df.explode("countries")
                  .dropna(subset=["countries"])
                  .rename(columns={"countries": "iso3"}))

    anchor = exploded["seendate"].max()
    cutoff = anchor - pd.Timedelta(days=primary_days)
    start  = exploded["seendate"].min()

    recent = exploded[exploded["seendate"] >= cutoff]
    prior  = exploded[exploded["seendate"] <  cutoff]

    recent_counts = recent.groupby("iso3").size().rename("recent_count")
    prior_counts  = prior.groupby("iso3").size().rename("prior_count")

    combined = pd.concat([recent_counts, prior_counts], axis=1).fillna(0)

    # normalise prior window to the same length as the primary window
    prior_days = max((cutoff - start).days, 1)
    combined["prior_norm"] = (combined["prior_count"] / prior_days) * primary_days

    # +0.5 smoothing: dampens 0→small jumps and prevents division-by-zero
    combined["spike_ratio"] = combined["recent_count"] / (combined["prior_norm"] + 0.5)

    result = (combined
              .loc[(combined["spike_ratio"] >= threshold) &
                   (combined["recent_count"] >= min_recent)]
              .sort_values("spike_ratio", ascending=False)
              .reset_index())

    print(f"  Gate 1: {len(result)} countries spiked ≥{threshold}× "
          f"(last {primary_days}d vs prior {prior_days}d, min {min_recent} recent articles)")
    return result


In [16]:
# Countries that sit on or adjacent to a major chokepoint.
# An event here is oil-relevant even if the country isn't a top producer.
# Values must match the PortWatch portname strings in PW_NAME so that
# PW_TO_SHORT (the reverse map) can resolve them for tanker data lookups.
CHOKEPOINT_ADJACENT = {
    "YEM": "Bab el-Mandeb Strait", "DJI": "Bab el-Mandeb Strait", "ERI": "Bab el-Mandeb Strait",
    "EGY": "Suez Canal",           "ISR": "Suez Canal",
    "IRN": "Strait of Hormuz",     "OMN": "Strait of Hormuz",
    "MYS": "Malacca Strait",       "SGP": "Malacca Strait",       "IDN": "Malacca Strait",
    "PAN": "Panama Canal",
    "TUR": "Turkish Straits",
    "DNK": "Danish Straits",       "SWE": "Danish Straits",
    # Indo-Pacific additions
    "TWN": "Taiwan Strait",                                         # Taiwan Strait corridor
    "PHL": "South China Sea",      "VNM": "South China Sea",       # South China Sea routing
}


def filter_oil_relevant(spikes_df, jodi_df, min_prod_kbd=100):
    """
    Gate 2: keep only spiking countries that are either meaningful crude producers
    (JODI latest value ≥ min_prod_kbd thousand barrels/day) or sit on a chokepoint.

    Adds 'relevance' and 'chokepoint' columns.
    """
    if spikes_df.empty:
        return pd.DataFrame()

    # derive producer set from JODI
    producers = set()
    if (not jodi_df.empty
            and "iso3" in jodi_df.columns
            and "OBS_VALUE" in jodi_df.columns):
        latest_prod = (jodi_df.sort_values("TIME_PERIOD")
                               .groupby("iso3")["OBS_VALUE"].last())
        producers = set(latest_prod[latest_prod >= min_prod_kbd].index)

    adjacent = set(CHOKEPOINT_ADJACENT)
    result = spikes_df[spikes_df["iso3"].isin(producers | adjacent)].copy()

    result["relevance"] = result["iso3"].apply(
        lambda x: ("producer+chokepoint" if x in producers and x in adjacent
                   else "producer"         if x in producers
                   else "chokepoint-adjacent")
    )
    result["chokepoint"] = result["iso3"].map(CHOKEPOINT_ADJACENT)

    print(f"  Gate 2: {len(result)} oil-relevant candidates "
          f"({len(producers)} known producers, {len(adjacent)} chokepoint-adjacent countries in scope)")
    return result.reset_index(drop=True)


In [17]:
def check_comovement(eia_df, portwatch_df, primary_days=30):
    """
    Gate 3: did Brent, WTI, or chokepoint tanker transits move over the primary window?

    Compares the mean value in the last primary_days to the equivalent prior window.
    A move of >2% in either direction counts as a market signal.

    Returns (signals dict, any_signal bool).
    """
    signals = {}

    # ── Price signals from EIA bulk ──────────────────────────
    for metric in ["brent_usd_bbl", "wti_usd_bbl"]:
        if eia_df.empty:
            break
        sub = (eia_df[eia_df["metric"] == metric]
               .sort_values("date")
               .dropna(subset=["value"]))
        if sub.empty:
            continue
        anchor = sub["date"].max()
        cutoff = anchor - pd.Timedelta(days=primary_days)
        recent = sub[sub["date"] >= cutoff]["value"]
        prior  = sub[sub["date"] <  cutoff].tail(primary_days)["value"]
        if recent.empty or prior.empty:
            continue
        move = recent.mean() - prior.mean()
        pct  = move / prior.mean() * 100 if prior.mean() else 0
        signals[metric] = {
            "direction": "up" if move > 0 else "down",
            "move":      round(move, 2),
            "move_pct":  round(pct, 1),
            "latest":    round(sub["value"].iloc[-1], 2),
        }

    # ── Transit signals from PortWatch ──────────────────────
    if not portwatch_df.empty and "portname" in portwatch_df.columns:
        for cp in portwatch_df["portname"].unique():
            sub = (portwatch_df[portwatch_df["portname"] == cp]
                   .sort_values("date")
                   .dropna(subset=["n_tanker"]))
            if sub.empty:
                continue
            anchor = sub["date"].max()
            cutoff = anchor - pd.Timedelta(days=primary_days)
            recent = sub[sub["date"] >= cutoff]["n_tanker"]
            prior  = sub[sub["date"] <  cutoff].tail(primary_days)["n_tanker"]
            if recent.empty or prior.empty:
                continue
            move = recent.mean() - prior.mean()
            pct  = move / prior.mean() * 100 if prior.mean() else 0
            signals[f"tanker_{cp}"] = {
                "direction": "up" if move > 0 else "down",
                "move":      round(move, 1),
                "move_pct":  round(pct, 1),
                "latest":    round(recent.mean(), 1),
            }

    # >2% move in either direction = market noticed something
    any_signal = any(abs(v["move_pct"]) > 2 for v in signals.values())

    print(f"  {'Signal detected' if any_signal else 'No significant market movement'} "
          f"(>{primary_days}d window, threshold >2%)")
    for name, v in signals.items():
        print(f"    {name}: {v['direction'].upper()} {abs(v['move_pct']):.1f}%  "
              f"(latest: {v['latest']})")

    return signals, any_signal


In [18]:
def detect_event_start(iso3_list, chokepoint_short=None):
    """
    Estimate when an event started from two independent signals:

      GDELT     -- first day article counts for these countries crossed 2x
                   their prior 21-day rolling average (the news spiked).
      PortWatch -- first day tanker volumes dropped >20% below their prior
                   30-day average (ships physically stopped moving).

    Returns a dict:
      gdelt_date      -- date of first news spike  (or None)
      portwatch_date  -- date of first tanker drop (or None)
      anchor_date     -- the date used as the event start anchor
      anchor_source   -- plain-English description of which signal drove it
    """
    result = {
        "gdelt_date":     None,
        "portwatch_date": None,
        "anchor_date":    None,
        "anchor_source":  None,
    }

    # Signal 1: GDELT — first day article count crossed 2x prior 21-day mean
    if "gdelt_discovery" in globals() and not gdelt_discovery.empty \
            and "seendate" in gdelt_discovery.columns:
        tagged = gdelt_discovery.copy()
        tagged["_ctry"] = tagged["title"].apply(extract_countries_from_title)
        mask = tagged["_ctry"].apply(lambda cs: any(c in cs for c in iso3_list))
        arts = tagged[mask].copy()
        if not arts.empty:
            arts["_day"] = arts["seendate"].dt.normalize()
            daily = arts.groupby("_day").size().sort_index()
            for i in range(7, len(daily)):
                prior_mean = daily.iloc[max(0, i - 21):i].mean()
                if prior_mean > 0 and daily.iloc[i] >= 2 * prior_mean:
                    result["gdelt_date"] = daily.index[i]
                    break
            if result["gdelt_date"] is None and not daily.empty:
                result["gdelt_date"] = daily.index[0]

    # Signal 2: PortWatch — first day 7-day smoothed volume < 80% of prior 30-day mean
    if chokepoint_short and "portwatch" in globals() and not portwatch.empty:
        _pw_full = PW_NAME.get(chokepoint_short)
        if _pw_full:
            s = (portwatch[portwatch["portname"] == _pw_full]
                 .sort_values("date").set_index("date")["n_tanker"])
            if len(s) >= 45:
                smoothed = s.rolling(7, min_periods=1).mean()
                for i in range(30, len(smoothed)):
                    baseline = smoothed.iloc[max(0, i - 30):i].mean()
                    if baseline > 0 and smoothed.iloc[i] < 0.8 * baseline:
                        result["portwatch_date"] = smoothed.index[i]
                        break

    # Anchor: prefer PortWatch (physical) over GDELT (perception)
    if result["portwatch_date"] is not None:
        result["anchor_date"]   = result["portwatch_date"]
        result["anchor_source"] = "PortWatch — first confirmed tanker volume drop"
    elif result["gdelt_date"] is not None:
        result["anchor_date"]   = result["gdelt_date"]
        result["anchor_source"] = "GDELT — first news spike (physical disruption not yet confirmed in data)"

    return result


def discover_events(primary_days=30, context_days=90, spike_threshold=5.0):
    """
    Event discovery pipeline -- three gates -> ranked candidate events.

    INVESTIGATE = passed all 3 gates (news spike + oil-relevant geography + market moved).
    WATCH       = passed Gates 1 & 2 but no market signal yet.

    Returns a DataFrame of candidates.
    """
    print("=" * 72)
    print("EVENT DISCOVERY")
    print("Primary window: {}d  |  Spike threshold: {}x".format(primary_days, spike_threshold))

    # Show date range of the news data we are working with
    if not gdelt_discovery.empty and "seendate" in gdelt_discovery.columns:
        d_min = gdelt_discovery["seendate"].min()
        d_max = gdelt_discovery["seendate"].max()
        print("News data covers: {} to {}  ({} articles)".format(
              d_min.strftime("%b %d, %Y") if pd.notna(d_min) else "?",
              d_max.strftime("%b %d, %Y") if pd.notna(d_max) else "?",
              len(gdelt_discovery)))
    print("=" * 72)

    # -- Gate 1 ---------------------------------------------------------------
    print("\nGate 1 -- Is any country getting significantly more news coverage than usual?")
    if gdelt_discovery.empty:
        print("  News data unavailable. Re-run the GDELT cell or check connection.")
        return pd.DataFrame()
    spikes = detect_spikes(gdelt_discovery, primary_days=primary_days, threshold=spike_threshold)

    if spikes.empty:
        print("  No unusual spikes found. Nothing to investigate.")
        return pd.DataFrame()

    # -- Gate 2 ---------------------------------------------------------------
    print("\nGate 2 -- Is the spiking country relevant to oil supply?")
    candidates = filter_oil_relevant(spikes, jodi)

    if candidates.empty:
        print("  No oil-relevant events found.")
        return pd.DataFrame()

    # -- Gate 3 ---------------------------------------------------------------
    print("\nGate 3 -- Did the oil market actually move?")
    market_signals, any_signal = check_comovement(eia, portwatch, primary_days=primary_days)

    candidates["classification"] = "WATCH"
    if any_signal:
        candidates["classification"] = "INVESTIGATE"

    brent = market_signals.get("brent_usd_bbl", {})
    candidates["brent_move_pct"]  = brent.get("move_pct")
    candidates["brent_direction"] = brent.get("direction")

    # -- Candidate summary table ----------------------------------------------
    print("\n" + "=" * 72)
    print("CANDIDATE EVENTS")
    print("Ranked by news spike size. INVESTIGATE = news spike + oil-relevant + market moved.")
    print("=" * 72)
    print("  {:<24}  {:<22}  {:>7}  {:>10}  {}".format(
          "Country", "Chokepoint", "Spike", "Oil Price 30d", "Status"))
    print("  {}  {}  {}  {}  {}".format("-"*24, "-"*22, "-"*7, "-"*10, "-"*12))
    for _, row in candidates.iterrows():
        try:
            cname = pycountry.countries.get(alpha_3=row["iso3"]).name
            if len(cname) > 20:
                cname = cname[:18] + ".."
        except Exception:
            cname = row["iso3"]
        label = "{} ({})".format(cname, row["iso3"])
        cp    = str(row["chokepoint"]) if pd.notna(row.get("chokepoint")) else "--"
        bpct  = ("{:+.1f}%".format(row["brent_move_pct"])
                 if pd.notna(row.get("brent_move_pct")) else "--")
        print("  {:<24}  {:<22}  {:>6.0f}x  {:>10}  {}".format(
              label, cp, row["spike_ratio"], bpct, row["classification"]))

    # -- Headlines + timeline per candidate -----------------------------------
    print("\n" + "=" * 72)
    print("WHAT IS DRIVING EACH CANDIDATE?")
    print("Read the headlines and coverage timeline to understand what happened")
    print("and when -- then decide which event you want to trace.")
    print("=" * 72)

    # Tag every article with the countries it mentions (done once)
    tagged = gdelt_discovery.copy()
    tagged["_countries"] = tagged["title"].apply(extract_countries_from_title)
    if "seendate" in tagged.columns:
        tagged["_day"] = tagged["seendate"].dt.normalize()

    for _, row in candidates.iterrows():
        iso3 = row["iso3"]
        try:
            cname = pycountry.countries.get(alpha_3=iso3).name
        except Exception:
            cname = iso3

        cp_note = ("  ->  {} chokepoint".format(row["chokepoint"])
                   if pd.notna(row.get("chokepoint")) else "")
        print("\n  {} ({})  |  {:.0f}x news spike  |  {}{}".format(
              cname, iso3, row["spike_ratio"], row["classification"], cp_note))

        mask = tagged["_countries"].apply(lambda cs: iso3 in cs)
        arts = tagged[mask].copy()

        # Coverage timeline: article count per day
        if not arts.empty and "_day" in arts.columns:
            daily   = arts.groupby("_day").size().sort_index()
            d_first = daily.index[0]
            d_last  = daily.index[-1]
            d_peak  = daily.idxmax()
            n_peak  = daily.max()
            span    = (d_last - d_first).days

            print("  Coverage: first seen {}  |  peak {}  ({} articles that day)  |  {} day span".format(
                  d_first.strftime("%b %d"),
                  d_peak.strftime("%b %d"),
                  n_peak,
                  span if span > 0 else "<1"))

            # Day-by-day bar chart (max bar width = 28 chars)
            max_count  = daily.max()
            bar_width  = 28
            print("  Daily article volume:")
            for day, count in daily.items():
                bar = int(round(count / max_count * bar_width)) if max_count else 0
                print("    {}  {}  {}".format(
                      day.strftime("%b %d"),
                      ("█" * bar).ljust(bar_width),
                      count))
        else:
            print("  No date information available for timeline.")

        # -- Event start estimate ---------------------------------------------
        _cp_raw   = row.get("chokepoint")
        _pw_short = {v: k for k, v in PW_NAME.items()}.get(str(_cp_raw)) \
                    if _cp_raw and not pd.isna(_cp_raw) else None
        _est = detect_event_start([iso3], chokepoint_short=_pw_short)
        print("  Event start estimate:")
        if _est["gdelt_date"] is not None:
            print("    News spike first detected   : {}  (GDELT)".format(
                  _est["gdelt_date"].strftime("%b %d, %Y")))
        else:
            print("    News spike first detected   : insufficient data")
        if _est["portwatch_date"] is not None:
            print("    Tanker volume drop detected : {}  (PortWatch)".format(
                  _est["portwatch_date"].strftime("%b %d, %Y")))
        elif _pw_short:
            print("    Tanker volume drop detected : no significant drop found in the data window")
        else:
            print("    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)")
        if _est["anchor_date"] is not None:
            print("    Baseline anchor             : {}  --  {}".format(
                  _est["anchor_date"].strftime("%b %d, %Y"), _est["anchor_source"]))
        else:
            print("    Baseline anchor             : could not be determined")
        _gap = None
        if _est["gdelt_date"] and _est["portwatch_date"]:
            _gap = (_est["portwatch_date"] - _est["gdelt_date"]).days
            if _gap > 0:
                print("    Fear premium window        : {} days between news spike and ships stopping".format(_gap))

        # Most recent headlines
        print("  Recent headlines:")
        recent = arts.sort_values("seendate", ascending=False).head(4)
        if recent.empty:
            print("    No headlines found.")
        else:
            for _, a in recent.iterrows():
                title    = str(a.get("title", "")).strip()
                if len(title) > 90:
                    title = title[:87] + "..."
                domain   = str(a.get("domain", "")).replace("www.", "")
                date_str = a["seendate"].strftime("%b %d") if pd.notna(a.get("seendate")) else ""
                print('    > "{}"'.format(title))
                print("      {} -- {}".format(domain, date_str))

    # -- Instructions ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("TO TRACE AN EVENT:")
    print("  1. Read the headlines and timeline above to pick the event you care about.")
    print("  2. Note the ISO-3 code for that country  (e.g. IRN, YEM, SAU).")
    print("  3. Scroll to the Control Panel cell below.")
    print('  4. Change  TRACE_TARGET = "auto"  to  TRACE_TARGET = "<ISO-3 code>"')
    print("  5. Re-run the Control Panel cell.")
    print("=" * 72)
    print()
    return candidates

candidates = discover_events()


EVENT DISCOVERY
Primary window: 30d  |  Spike threshold: 5.0x
News data covers: Jun 23, 2026 to Jul 22, 2026  (500 articles)

Gate 1 -- Is any country getting significantly more news coverage than usual?


  Gate 1: 12 countries spiked ≥5.0× (last 30d vs prior 1d, min 3 recent articles)

Gate 2 -- Is the spiking country relevant to oil supply?
  Gate 2: 8 oil-relevant candidates (35 known producers, 17 chokepoint-adjacent countries in scope)

Gate 3 -- Did the oil market actually move?
  Signal detected (>30d window, threshold >2%)
    brent_usd_bbl: DOWN 28.2%  (latest: 81.62)
    wti_usd_bbl: DOWN 25.2%  (latest: 79.2)
    tanker_Suez Canal: UP 9.2%  (latest: 16.5)
    tanker_Panama Canal: UP 12.3%  (latest: 15.4)
    tanker_Bab el-Mandeb Strait: DOWN 8.0%  (latest: 13.2)
    tanker_Malacca Strait: UP 5.7%  (latest: 72.9)
    tanker_Strait of Hormuz: UP 457.9%  (latest: 9.5)
    tanker_Taiwan Strait: DOWN 6.6%  (latest: 46.0)
    tanker_Lombok Strait: UP 2.7%  (latest: 7.8)
    tanker_Sunda Strait: DOWN 6.5%  (latest: 11.8)

CANDIDATE EVENTS
Ranked by news spike size. INVESTIGATE = news spike + oil-relevant + market moved.
  Country                   Chokepoint                Spike  Oi


  Iran, Islamic Republic of (IRN)  |  280x news spike  |  INVESTIGATE  ->  Strait of Hormuz chokepoint
  Coverage: first seen Jun 23  |  peak Jul 22  (75 articles that day)  |  29 day span
  Daily article volume:
    Jun 23  ████████████████████████      65
    Jul 22  ████████████████████████████  75


  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : Apr 26, 2026  (PortWatch)
    Baseline anchor             : Apr 26, 2026  --  PortWatch — first confirmed tanker volume drop
  Recent headlines:
    > "US will destroy bridge or power plant for each Iranian attack in strait – Trump"
      greenocktelegraph.co.uk -- Jul 22
    > "Trump says US will destroy a bridge or power plant for each Iranian attack in the Strai..."
      wbay.com -- Jul 22
    > "US will destroy bridge or power plant for each Iranian attack in strait – Trump"
      swindonadvertiser.co.uk -- Jul 22
    > "Rubio Says Iran - Backed Houthi Red Sea Blockade  Problematic"
      iraqsun.com -- Jul 22

  United States (USA)  |  154x news spike  |  INVESTIGATE
  Coverage: first seen Jun 23  |  peak Jun 23  (40 articles that day)  |  29 day span
  Daily article volume:
    Jun 23  ████████████████████████████  40
    Jul 22  ██████████████████████████    37


  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)
    Baseline anchor             : Jun 23, 2026  --  GDELT — first news spike (physical disruption not yet confirmed in data)
  Recent headlines:
    > "US will destroy bridge or power plant for each Iranian attack in strait – Trump"
      greenocktelegraph.co.uk -- Jul 22
    > "Trump says US will destroy a bridge or power plant for each Iranian attack in the Strai..."
      wbay.com -- Jul 22
    > "US trial of Venezuela Maduro set for June 2027"
      bssnews.net -- Jul 22
    > "Segro board U - turns on £14bn takeover bid by US rival Prologis"
      aol.co.uk -- Jul 22

  Yemen (YEM)  |  42x news spike  |  INVESTIGATE  ->  Bab el-Mandeb Strait chokepoint
  Coverage: first seen Jul 22  |  peak Jul 22  (21 articles that day)  |  <1 day span
  Daily article volume:
    Jul 22  ████████████████████████████  21


...

  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)
    Baseline anchor             : Jun 23, 2026  --  GDELT — first news spike (physical disruption not yet confirmed in data)
  Recent headlines:
    > "Israel , the Only Middle East Nuclear Weapons Power , Fears US - Saudi Deal"
      newsweek.com -- Jul 22
    > "Trump Secret Plan to Allow Saudi Arabia to Enrich Nuclear Fuel Leaks"
      thedailybeast.com -- Jul 22
    > "US , Pakistan condemn Houthi Red Sea threats to shipping , Saudi Arabia"
      thetimes-tribune.com -- Jul 22
    > "Iran War Causes Massive Disruptions in Jordan , Kuwait , Saudi Arabia"
      breitbart.com -- Jul 22

  India (IND)  |  10x news spike  |  INVESTIGATE
  Coverage: first seen Jun 23  |  peak Jun 23  (4 articles that day)  |  29 day span
  Daily article volume:
    Jun 23  ████████████████████████████  4
    Jul 22  ███████                       1


  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)
    Baseline anchor             : Jun 23, 2026  --  GDELT — first news spike (physical disruption not yet confirmed in data)
  Recent headlines:
    > "Temasek Investments : Temasek widens scope of India investments"
      timesofindia.indiatimes.com -- Jul 22
    > "India and Qatar stand in solidarity : PM Modi after Qatar Emir condoles death of 12 Ind..."
      moneycontrol.com -- Jun 23
    > "No real alternative to Middle Eastern LPG for India : S & P Global Pulkit Agarwal"
      sierraleonetimes.com -- Jun 23
    > "Iran - U . S . MoU has had a positive impact on energy , fertiliser flow to India : MEA"
      thehindu.com -- Jun 23

  Israel (ISR)  |  8x news spike  |  INVESTIGATE  ->  Suez Canal chokepoint
  Coverage: first seen Jun 23  |  peak Jun 23  (2 articles that day)  |  29 day span
  Daily article volume:
    Jun 23

  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : May 02, 2026  (PortWatch)
    Baseline anchor             : May 02, 2026  --  PortWatch — first confirmed tanker volume drop
  Recent headlines:
    > "Israel , the Only Middle East Nuclear Weapons Power , Fears US - Saudi Deal"
      newsweek.com -- Jul 22
    > "Missão da UE emite alerta a navios ligados a Israel , EUA e Arábia Saudita , e grupo di..."
      oglobo.globo.com -- Jul 22
    > "War On Iran : – Israel , And Trump Himself , Are Hindering Conflict Solution – Moon of ..."
      moonofalabama.org -- Jun 23
    > "Israel vuelve a matar en Líbano antes de una nueva ronda de contactos con el Gobierno d..."
      elnortedecastilla.es -- Jun 23

  Venezuela, Bolivarian Republic of (VEN)  |  8x news spike  |  INVESTIGATE
  Coverage: first seen Jun 23  |  peak Jul 22  (3 articles that day)  |  29 day span
  Daily article volume:
    Jun 23  █████████                    

  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)
    Baseline anchor             : Jun 23, 2026  --  GDELT — first news spike (physical disruption not yet confirmed in data)
  Recent headlines:
    > "US trial of Venezuela Maduro set for June 2027"
      bssnews.net -- Jul 22
    > "US court fixes June 2027 trial for Venezuela ousted Maduro"
      vanguardngr.com -- Jul 22
    > "Venezuela ousted leader Maduro back in U . S . court"
      thehindu.com -- Jul 22
    > "Venezuela post - Maduro : will oil majors invest again ?"
      power-technology.com -- Jun 23

  Qatar (QAT)  |  6x news spike  |  INVESTIGATE
  Coverage: first seen Jun 23  |  peak Jun 23  (3 articles that day)  |  <1 day span
  Daily article volume:
    Jun 23  ████████████████████████████  3


  Event start estimate:
    News spike first detected   : Jun 23, 2026  (GDELT)
    Tanker volume drop detected : N/A (no chokepoint linked to this candidate)
    Baseline anchor             : Jun 23, 2026  --  GDELT — first news spike (physical disruption not yet confirmed in data)
  Recent headlines:
    > "How Qatar made itself impossible to punish"
      ynetnews.com -- Jun 23
    > "India and Qatar stand in solidarity : PM Modi after Qatar Emir condoles death of 12 Ind..."
      moneycontrol.com -- Jun 23
    > "Qatar Says Ras Laffan Blast Will Not Affect LNG Exports"
      rigzone.com -- Jun 23

TO TRACE AN EVENT:
  1. Read the headlines and timeline above to pick the event you care about.
  2. Note the ISO-3 code for that country  (e.g. IRN, YEM, SAU).
  3. Scroll to the Control Panel cell below.
  4. Change  TRACE_TARGET = "auto"  to  TRACE_TARGET = "<ISO-3 code>"
  5. Re-run the Control Panel cell.



## 8 · War-game trace — five-layer cascade

`run_top_candidate()` picks the top-ranked event from §9's discovery pipeline and runs
`trace_event()` on it automatically. **No chokepoint is hardcoded** — the geography comes
from what the data surfaced.

The five layers, in the order the method demands:

1. **① DETECT** — GDELT news volume. The event tells you *where* to look; it does not prove supply moved.
2. **② VERIFY GATE** — IMF PortWatch tanker transits + EIA crude stocks. Separates a **real physical disruption** (transits fell) from a **risk premium** (fear/price, no physical cut). Only a confirmed drop justifies reading the rest as live.
3. **③ SUPPLY & PRICE** — crude at risk (JODI), Brent/WTI move (EIA), geopolitical risk (GPR).
4. **④ JET FUEL** — US Gulf Coast jet-fuel spot (EIA); crack spread over Brent.
5. **⑤ DOWNSTREAM** — exposed crude importers (Comtrade) → airfields in those countries (OurAirports).


In [19]:
# Which producing countries mainly ship through each chokepoint (ISO-3).
# Editable — drawn from EIA chokepoint analysis.
CHOKEPOINT_EXPORTERS = {
    "Hormuz":      ["SAU","IRN","IRQ","KWT","ARE","QAT","BHR"],
    "Malacca":     ["SAU","ARE","KWT","IRQ"],
    "Suez":        ["SAU","IRQ","RUS","KAZ"],
    "BabelMandeb": ["SAU","IRQ","RUS","KAZ"],
    "Panama":      ["USA","COL","ECU"],
    # Indo-Pacific alternative and secondary corridors
    "Lombok":      ["SAU","ARE","KWT","IRQ"],          # alternative to Malacca; same origin oil
    "Sunda":       ["SAU","ARE","KWT","IRQ"],          # second Malacca alternative
    "Taiwan":      ["SAU","ARE","KWT","IRQ","RUS","AUS","MYS","IDN"],  # NE Asia-bound crude
    "SCS":         ["SAU","ARE","IRQ","KWT","IRN","RUS","MYS","IDN"],  # South China Sea corridor
}

# Reverse map: PortWatch full name -> trace_event() short name
PW_TO_SHORT = {v: k for k, v in PW_NAME.items()}


# -- small helpers -------------------------------------------------------------
def _series(metric):
    if "eia" not in globals() or eia.empty:
        return pd.DataFrame(columns=["date", "value"])
    return eia[eia["metric"] == metric].sort_values("date")

def _move(metric, n):
    s = _series(metric)
    if len(s) <= n:
        return (s["value"].iloc[-1] if len(s) else None, None)
    now, then = s["value"].iloc[-1], s["value"].iloc[-1 - n]
    return now, (now / then - 1) * 100 if then else None

def _news_spike(news, recent_days=3, window_days=21):
    if news is None or news.empty or "seendate" not in news:
        return None
    d = news.dropna(subset=["seendate"]).copy()
    if d.empty:
        return None
    d["day"] = d["seendate"].dt.floor("D")
    end     = d["day"].max()
    span    = d[d["day"] > end - pd.Timedelta(days=window_days)]
    per_day = span.groupby("day").size()
    recent  = per_day[per_day.index > end - pd.Timedelta(days=recent_days)].mean()
    base    = per_day[per_day.index <= end - pd.Timedelta(days=recent_days)].mean()
    recent  = 0 if pd.isna(recent) else recent
    base    = 0 if pd.isna(base)   else base
    ratio   = (recent / base) if base else (float("inf") if recent else 0)
    return {"recent": recent, "base": base, "ratio": ratio, "spike": ratio >= 1.5}

def _gate(pw_name, recent_days=7, drop_last=1, pre_lo=14, pre_hi=45, thresh=0.7):
    if not pw_name or "portwatch" not in globals() or portwatch.empty:
        return None
    s = (portwatch[portwatch["portname"] == pw_name]
         .sort_values("date").set_index("date")["n_tanker"])
    if len(s) < pre_hi + recent_days:
        return {"insufficient": True, "n": len(s)}
    end    = s.index.max()
    r_end  = end - pd.Timedelta(days=drop_last)
    recent = s.loc[r_end - pd.Timedelta(days=recent_days) : r_end].mean()
    pre    = s.loc[r_end - pd.Timedelta(days=pre_hi)      : r_end - pd.Timedelta(days=pre_lo)].mean()
    ratio  = (recent / pre) if pre else float("nan")
    return {"pw_name": pw_name, "recent": recent, "pre": pre, "ratio": ratio,
            "disrupted": (ratio < thresh)}

def _country_label(iso3):
    """Return 'Full Name (ISO3)' using pycountry, or just ISO3 if lookup fails."""
    try:
        name = pycountry.countries.get(alpha_3=iso3).name
        if len(name) > 20:
            name = name[:18] + ".."
        return "{} ({})".format(name, iso3)
    except Exception:
        return iso3


def trace_event(chokepoint, date, days_of_cover=None):
    exporters = CHOKEPOINT_EXPORTERS.get(chokepoint, [])
    pw_name   = PW_NAME.get(chokepoint)
    head      = []
    print("=" * 80)
    print("WAR-GAME TRACE  .  {}  .  as of {}".format(chokepoint, date))
    print("=" * 80)

    # -- Data vintage summary ------------------------------------------------
    # Show exactly what time period each source is reporting from so the
    # analyst knows whether they are comparing apples to apples.
    _today = pd.Timestamp.today().normalize()

    def _vintage(df, date_col="date", freq="daily"):
        if df is None or (hasattr(df, "empty") and df.empty):
            return "no data"
        try:
            col = next((c for c in [date_col, "date", "seendate", "TIME_PERIOD"]
                        if c in df.columns), None)
            if col is None:
                return "unknown"
            latest = pd.to_datetime(df[col], errors="coerce").max()
            if pd.isna(latest):
                return "unknown"
            age = (_today - latest.normalize()).days
            age_str = "today" if age == 0 else "{} day{} old".format(age, "s" if age != 1 else "")
            stale = "  ⚠" if (
                (freq == "daily"   and age > 5)  or
                (freq == "weekly"  and age > 10) or
                (freq == "monthly" and age > 45)
            ) else ""
            return "{}{}{} ({})".format(
                latest.strftime("%b %d, %Y") if freq == "daily" else latest.strftime("%b %Y"),
                stale, "", age_str)
        except Exception:
            return "unknown"

    print("\nDATA VINTAGE  (what time period each source is actually reporting)")
    print("  Sources have different publication schedules -- numbers below are NOT all from")
    print("  the same date. Read the vintage for each before drawing conclusions.")
    print()

    # News
    if "gdelt_discovery" in globals() and not gdelt_discovery.empty and "seendate" in gdelt_discovery.columns:
        d = gdelt_discovery["seendate"].max()
        print("  {:<28}  {}".format("News (GDELT)", d.strftime("%b %d, %Y") + "  (near real-time)"))
    else:
        print("  {:<28}  {}".format("News (GDELT)", "no data"))

    # PortWatch
    if "portwatch" in globals() and not portwatch.empty and "date" in portwatch.columns:
        print("  {:<28}  {}".format("Tanker traffic (PortWatch)", _vintage(portwatch, "date", "daily")))
    else:
        print("  {:<28}  {}".format("Tanker traffic (PortWatch)", "no data"))

    # EIA prices
    _brent = _series("brent_usd_bbl")
    if not _brent.empty:
        print("  {:<28}  {}".format("Oil prices (EIA)", _vintage(_brent, "date", "daily")))
    else:
        print("  {:<28}  {}".format("Oil prices (EIA)", "no data"))

    # EIA jet fuel
    _jet = _series("jet_fuel_usd_gal")
    if not _jet.empty:
        print("  {:<28}  {}".format("Jet fuel price (EIA)", _vintage(_jet, "date", "daily")))

    # EIA stocks
    _stk = _series("us_crude_stocks_kbbl")
    if not _stk.empty:
        print("  {:<28}  {}".format("US crude stockpile (EIA)", _vintage(_stk, "date", "weekly")))

    # GPR
    if "gpr" in globals() and not gpr.empty and "date" in gpr.columns:
        print("  {:<28}  {}".format("Geopolitical risk (GPR)", _vintage(gpr, "date", "monthly")))

    # Pink Sheet
    if "pink_prices" in globals() and not pink_prices.empty and "date" in pink_prices.columns:
        print("  {:<28}  {}".format("Commodity prices (Pink Sheet)", _vintage(pink_prices, "date", "monthly")))

    # JODI production
    if "jodi" in globals() and not jodi.empty and "TIME_PERIOD" in jodi.columns:
        print("  {:<28}  {}".format("Oil production (JODI)", _vintage(jodi, "TIME_PERIOD", "monthly")))

    # Comtrade
    if "crude_imports" in globals() and not crude_imports.empty:
        if "refYear" in crude_imports.columns and "refMonth" in crude_imports.columns:
            _yr = int(crude_imports["refYear"].iloc[0])
            _mo = int(crude_imports["refMonth"].iloc[0])
            _ct_date = pd.Timestamp(year=_yr, month=_mo, day=1)
            _ct_age  = (_today - _ct_date).days
            print("  {:<28}  {} ({} days old)".format(
                  "Trade flows (Comtrade)",
                  _ct_date.strftime("%b %Y"), _ct_age))
        else:
            print("  {:<28}  {}".format("Trade flows (Comtrade)", "date unknown"))

    # World Bank
    if "worldbank" in globals() and not worldbank.empty:
        print("  {:<28}  {}".format("Economic data (World Bank)", "most recent annual (1–2 yr lag)"))

    print()
    print("  ⚠ = older than expected for that source's publication schedule")
    print("-" * 72)

    # -- Event start estimate -------------------------------------------------
    _est = detect_event_start(exporters, chokepoint_short=chokepoint)
    print("\nEVENT START ESTIMATE")
    print("   The tool looks for two signals to pin down when this disruption began.")
    print()
    if _est["gdelt_date"] is not None:
        print("   News spike first detected   : {}  (GDELT)".format(
              _est["gdelt_date"].strftime("%b %d, %Y")))
        print("   This is when global news coverage of the affected region first")
        print("   crossed double its normal level — the market was paying attention.")
    else:
        print("   News spike first detected   : insufficient data")
    print()
    if _est["portwatch_date"] is not None:
        print("   Tanker volume drop detected : {}  (PortWatch)".format(
              _est["portwatch_date"].strftime("%b %d, %Y")))
        print("   This is when ship traffic first dropped >20% below its prior 30-day")
        print("   average — ships physically stopped moving through the chokepoint.")
    else:
        print("   Tanker volume drop detected : no significant drop found in the data window")
    print()
    if _est["anchor_date"] is not None:
        print("   Baseline anchor : {}  --  {}".format(
              _est["anchor_date"].strftime("%b %d, %Y"), _est["anchor_source"]))
        print("   All before/after comparisons below are anchored to this date.")
    else:
        print("   Baseline anchor : could not be determined from available data")
    if _est["gdelt_date"] and _est["portwatch_date"]:
        _gap = (_est["portwatch_date"] - _est["gdelt_date"]).days
        if _gap > 0:
            print()
            print("   Fear premium window : {} day(s) elapsed between the news spike and ships".format(_gap))
            print("   actually stopping.  During this window oil prices moved on fear, not yet")
            print("   on a real supply cut.")
    print("-" * 72)

    # -- (1) News signal -------------------------------------------------------
    print("\n(1) NEWS SIGNAL   (how much is this event dominating media coverage?)")

    # Build per-day article counts for countries linked to this chokepoint
    _gdelt = gdelt_discovery if "gdelt_discovery" in globals() else pd.DataFrame()
    _timeline_countries = exporters + list(CHOKEPOINT_ADJACENT.get(
        [k for k, v in CHOKEPOINT_ADJACENT.items() if v and chokepoint in v][0]
        if any(chokepoint in str(v) for v in CHOKEPOINT_ADJACENT.values()) else "",
        []
    ) if False else [])  # exporters is sufficient for timeline

    if not _gdelt.empty and "title" in _gdelt.columns and "seendate" in _gdelt.columns:
        _tagged = _gdelt.copy()
        _tagged["_ctry"] = _tagged["title"].apply(extract_countries_from_title)
        _mask   = _tagged["_ctry"].apply(lambda cs: any(c in cs for c in exporters))
        _arts   = _tagged[_mask].copy()

        if not _arts.empty:
            _arts["_day"] = _arts["seendate"].dt.normalize()
            _daily  = _arts.groupby("_day").size().sort_index()
            _first  = _daily.index[0]
            _last   = _daily.index[-1]
            _peak   = _daily.idxmax()
            _npeak  = int(_daily.max())
            _span   = (_last - _first).days

            print("   Coverage period : {} to {}  ({} day span)".format(
                  _first.strftime("%b %d, %Y"),
                  _last.strftime("%b %d, %Y"),
                  _span if _span > 0 else "<1"))
            print("   First appeared  : {}".format(_first.strftime("%b %d, %Y")))
            print("   Peak coverage   : {}  ({} articles that day)".format(
                  _peak.strftime("%b %d, %Y"), _npeak))

            if _span >= 1:
                trend = "still escalating" if _daily.iloc[-1] >= _daily.iloc[-2] else "cooling off"
                print("   Trend           : {}".format(trend))

            _bw  = 30
            _max = _daily.max()
            print("   Daily article volume (mentions of exporter countries):")
            for _day, _cnt in _daily.items():
                _bar = int(round(_cnt / _max * _bw)) if _max else 0
                print("     {}  {}  {}".format(
                      _day.strftime("%b %d"),
                      ("█" * _bar).ljust(_bw),
                      _cnt))

            head.append("news coverage from {} -- peak {}".format(
                         _first.strftime("%b %d"), _peak.strftime("%b %d")))
        else:
            print("   No articles found mentioning the affected exporter countries.")
    else:
        print("   No news data available.")

    sig = _news_spike(_gdelt) if not _gdelt.empty else None
    if sig:
        if sig["base"] == 0:
            print("   Spike ratio     : unreliable (no historical baseline -- API cap hit)")
        else:
            flag = "ELEVATED" if sig["spike"] else "normal"
            print("   Spike ratio     : {:.1f}x  [{}]".format(sig["ratio"], flag))
    print("   Note: news signal shows WHERE to look -- it does not confirm supply was disrupted.")

    # -- (2) Physical disruption check -----------------------------------------
    print("\n(2) PHYSICAL DISRUPTION CHECK   (did ships actually stop moving?)")
    print("   Source: IMF PortWatch -- counts tanker ships passing through the chokepoint each day.")
    g = _gate(pw_name)
    if g is None:
        print("   No PortWatch data available for this chokepoint.")
    elif g.get("insufficient"):
        print("   Only {} days of transit data -- not enough history to judge.".format(g["n"]))
    else:
        pct = g["ratio"] * 100
        print("   Chokepoint: {}".format(g["pw_name"]))
        print("   Tanker ships in the last 7 days   : {:.1f} per day".format(g["recent"]))
        print("   Tanker ships before the event     : {:.1f} per day  "
              "(average from 14 to 45 days ago)".format(g["pre"]))
        print("   Current traffic as % of normal    : {:.0f}%".format(pct))
        if g["disrupted"]:
            verdict = "CONFIRMED PHYSICAL DISRUPTION -- tanker traffic has collapsed"
            head.append("tanker traffic at {:.0f}% of normal -- physical disruption confirmed".format(pct))
        else:
            verdict = "NO PHYSICAL DISRUPTION -- traffic is normal; any price move is fear-driven"
            head.append("tanker traffic normal -- price move is risk premium, not a real supply cut")
        print("   Result: {}".format(verdict))

    st = _series("us_crude_stocks_kbbl").tail(8)
    if len(st) >= 2:
        chg   = st["value"].iloc[-1] - st["value"].iloc[0]
        trend = "falling" if chg < 0 else "rising"
        print()
        print("   US crude oil in storage: {:,.0f} thousand barrels  "
              "({} by {:+,.0f} over {} weeks)".format(
              st["value"].iloc[-1], trend, chg, len(st) - 1))
        if chg < 0:
            print("   Falling stockpiles indicate that supply is not keeping up with demand.")

    if g and g.get("disrupted") is True:
        print()
        print("   >>> Disruption confirmed -- treat the supply and price numbers below as a LIVE event.")
    elif g and g.get("disrupted") is False:
        print()
        print("   >>> No confirmed disruption -- the analysis below is a WHAT-IF, not a live event.")

    # Singapore refinery note — Malacca/Lombok/Sunda only
    if chokepoint in ("Malacca", "Lombok", "Sunda"):
        print()
        print("   SINGAPORE REFINERY ALERT")
        print("   Singapore is the Indo-Pacific's primary jet fuel refining hub.")
        print("   Middle East crude arrives via this chokepoint, is refined in Singapore,")
        print("   and exported as jet fuel to Japan, South Korea, Taiwan, and Australia.")
        print("   A sustained closure cuts regional jet fuel production directly.")
        sgp_ref_val = None
        if "jodi_refinery" in globals() and not jodi_refinery.empty                 and "iso3" in jodi_refinery.columns:
            _sgp = jodi_refinery[jodi_refinery["iso3"] == "SGP"].sort_values("TIME_PERIOD")
            if not _sgp.empty:
                _latest_ref = _sgp.iloc[-1]
                sgp_ref_val = _latest_ref["OBS_VALUE"]
                print("   Singapore refinery throughput ({}) : {:,.0f} thousand bbl/day  (JODI)".format(
                      _latest_ref.get("TIME_PERIOD", "?"), sgp_ref_val))
        if sgp_ref_val is None:
            print("   Singapore refinery throughput: ~1,400 thousand bbl/day (2023 avg)")
            print("   Load the JODI file for a live figure.")


    # -- (3) Supply and price --------------------------------------------------
    print("\n(3) CRUDE OIL SUPPLY & PRICE")
    _brent_s = _series("brent_usd_bbl")
    if not _brent_s.empty:
        _eia_date = _brent_s["date"].max()
        _age_d    = int((pd.Timestamp.today() - _eia_date).days)
        _stale    = "  WARNING: DATA IS STALE -- prices shown are not current" if _age_d > 5 else ""
        print("   [Price data last updated: {}  ({} days ago){}]".format(
              _eia_date.date(), _age_d, _stale))

    if "crude_prod_latest" in country.columns:
        at_risk = country[country["iso3"].isin(exporters)]["crude_prod_latest"].sum()
        world   = country["crude_prod_latest"].sum()
        share   = " ({:.0f}% of total world output)".format(at_risk/world*100) if world else ""
        print()
        print("   Daily crude oil output from affected exporters: {:,.0f} thousand barrels/day{}".format(
              at_risk, share))
        print("   This is the volume that normally ships through the disrupted chokepoint.")
        print("   Note: production data has a ~2 month publication lag -- actual output may differ.")
        head.append("{:,.0f} thousand barrels/day of supply at risk".format(at_risk))

    print()
    # Brent = the main international oil price benchmark (crude from the North Sea).
    # WTI   = the US domestic benchmark (West Texas Intermediate); usually $2-4 below Brent.
    for m, lab in [("brent_usd_bbl", "Brent (global oil price benchmark)"),
                   ("wti_usd_bbl",   "WTI   (US domestic oil price benchmark)")]:
        now, wk = _move(m, 5)
        if now is not None:
            mv = "  ({:+.1f}% vs one week ago)".format(wk) if wk is not None else ""
            print("   {}: ${:,.2f} per barrel{}".format(lab, now, mv))
            if "Brent" in lab and wk is not None:
                head.append("Brent oil price {:+.1f}% over the past week".format(wk))

    if "gpr" in globals() and not gpr.empty:
        cur = gpr["GPR"].iloc[-1]
        pct = (gpr["GPR"] < cur).mean() * 100
        print()
        print("   Geopolitical Risk Index: {:.0f}  "
              "(higher than {:.0f}% of all months ever recorded)".format(cur, pct))
        print("   Measures how much global conflict and political tension is in the news.")

    # -- (4) Jet fuel ----------------------------------------------------------
    print("\n(4) JET FUEL PRICE")
    jet, jwk  = _move("jet_fuel_usd_gal", 5)
    brent_ser = _series("brent_usd_bbl")
    if jet is not None:
        jet_bbl = jet * 42   # 1 barrel = 42 US gallons
        print("   US Gulf Coast jet fuel: ${:,.3f} per gallon  "
              "(${:,.2f} per barrel -- 1 barrel = 42 gallons)".format(jet, jet_bbl))
        if jwk is not None:
            print("   Change vs one week ago: {:+.1f}%".format(jwk))
        if len(brent_ser):
            crack = jet_bbl - brent_ser["value"].iloc[-1]
            print("   Premium over raw crude oil: ${:,.2f} per barrel".format(crack))
            print("   (Jet fuel costs more than crude because it must be refined.")
            print("    A rising premium means refining capacity or distribution is under strain.)")
    else:
        print("   Jet fuel price data not available.")

    # -- (5) Countries and airports at risk ------------------------------------
    print("\n(5) COUNTRIES AND AIRPORTS AT RISK")
    print()
    print("   How this section works:")
    print("   1. Find which countries import crude oil from the affected exporters")
    print("      (using UN Comtrade bilateral trade data)")
    print("   2. Score each country on how exposed it is:")
    print("      - What fraction of its crude imports come from the disrupted exporters?")
    print("      - Does it produce enough oil domestically to offset a supply cut?")
    print("   3. Count the airports in each country -- those airports face potential fuel supply risk")
    print()
    print("   Risk tiers:")
    print("   CRITICAL -- imports most of its crude from the disrupted exporters")
    print("               AND produces little oil domestically (no fallback supply)")
    print("   ELEVATED -- moderate exposure to disrupted supply")
    print("               OR limited domestic production to buffer a supply cut")
    print("   WATCH    -- some exposure but domestic production or low share provides cover")

    bilateral_exp = pd.DataFrame()
    if (not crude_imports.empty
            and "partner_iso3" in crude_imports.columns
            and "iso3" in crude_imports.columns):
        from_exp = (crude_imports[crude_imports["partner_iso3"].isin(exporters)]
                    .groupby("iso3")["primaryValue"].sum()
                    .rename("import_from_exporters"))
        total_imp = (crude_imports.groupby("iso3")["primaryValue"].sum()
                     .rename("import_total"))
        exp_df = pd.concat([from_exp, total_imp], axis=1).fillna(0)
        exp_df["import_dep_share"] = (
            exp_df["import_from_exporters"]
            / exp_df["import_total"].replace(0, float("nan"))
        )
        bilateral_exp = exp_df[exp_df["import_from_exporters"] > 0].reset_index()
        if "refYear" in crude_imports.columns and "refMonth" in crude_imports.columns:
            _ct_yr = crude_imports["refYear"].iloc[0]
            _ct_mo = crude_imports["refMonth"].iloc[0]
            print()
            print("   Trade data: {}-{:02d}  (Comtrade free tier, capped at 2,500 rows)".format(
                  int(_ct_yr), int(_ct_mo)))
            print("   WARNING: Large importers like China, India, and South Korea may be missing")
            print("   because the free API cap cuts off before reaching all bilateral records.")

    if bilateral_exp.empty and "crude_import_value" in country.columns:
        fallback = (country.dropna(subset=["crude_import_value"])
                           .sort_values("crude_import_value", ascending=False).head(15))
        bilateral_exp = fallback[["iso3", "crude_import_value"]].copy().rename(
            columns={"crude_import_value": "import_from_exporters"}
        )
        bilateral_exp["import_total"]     = bilateral_exp["import_from_exporters"]
        bilateral_exp["import_dep_share"] = float("nan")
        print("   (Using total import value as fallback -- no bilateral source breakdown available)")

    if bilateral_exp.empty:
        print("   No trade data available to identify exposed importers.")
    else:
        ctx_cols = [c for c in ["crude_prod_latest", "oil_rents_pct_gdp", "gdp_usd"]
                    if c in country.columns]
        exp = bilateral_exp.merge(country[["iso3"] + ctx_cols], on="iso3", how="left")
        exp["crude_prod_latest"] = exp["crude_prod_latest"].fillna(0)

        TIER_RANK = {"CRITICAL": 0, "ELEVATED": 1, "WATCH": 2}

        def _classify_row(dep, prod):
            if pd.isna(dep):
                return "WATCH" if prod >= 500 else "ELEVATED"
            if dep >= 0.25 and prod < 100:
                return "CRITICAL"
            if dep >= 0.10 or (dep >= 0.05 and prod < 500):
                return "ELEVATED"
            return "WATCH"

        exp["tier"] = [
            _classify_row(r["import_dep_share"], r["crude_prod_latest"])
            for _, r in exp.iterrows()
        ]
        exp["tier_rank"] = exp["tier"].map(TIER_RANK)

        ap_counts = (airports.groupby("iso3")
                             .agg(total_airports=("ident", "count"),
                                  large_airports=("type", lambda x: (x == "large_airport").sum()))
                             .reset_index())
        exp = exp.merge(ap_counts, on="iso3", how="left")
        exp["large_airports"] = exp["large_airports"].fillna(0).astype(int)
        exp["total_airports"] = exp["total_airports"].fillna(0).astype(int)
        exp = (exp.sort_values(["tier_rank", "import_from_exporters"], ascending=[True, False])
                  .reset_index(drop=True)
                  .drop(columns=["tier_rank"]))

        us_doc = None
        if "eia" in globals() and not eia.empty:
            stk = eia[eia["metric"] == "us_crude_stocks_kbbl"].sort_values("date")
            imp = eia[eia["metric"] == "us_crude_imports_kbd"].sort_values("date")
            if not stk.empty and not imp.empty and imp["value"].iloc[-1] > 0:
                us_doc = stk["value"].iloc[-1] / imp["value"].iloc[-1]

        W0, W1, W2, W3, W4, W5 = 26, 22, 24, 24, 14, 12

        print()
        print("   Column guide:")
        print("     'Crude Bought from Affected Exporters ($M)'")
        print("       Dollar value of crude oil imported from the exporters that ship through")
        print("       the disrupted chokepoint, in the most recent Comtrade period.")
        print("     '% of This Country\'s Total Crude Imports'")
        print("       How large a share of their total crude purchases that represents.")
        print("       Higher percentage = more dependent on the disrupted supply.")
        print("     'Own Crude Production (thousand barrels/day)'")
        print("       How much crude oil this country produces itself.")
        print("       A country that produces a lot can offset an import cut.")
        print("     'Large Airports'")
        print("       Airports capable of handling commercial jet aircraft (OurAirports data).")

        hdr  = ("  {:<{w0}}  {:>{w1}}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}".format(
                "Country", "Crude Bought from", "% of This Country's", "Own Crude Production",
                "Large", "All", w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))
        hdr2 = ("  {:<{w0}}  {:>{w1}}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}".format(
                "", "Affected Exp. ($M)", "Total Crude Imports", "(thousand bbl/day)",
                "Airports", "Airports", w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))
        sep  = "  " + "  ".join(["-"*W0, "-"*W1, "-"*W2, "-"*W3, "-"*W4, "-"*W5])

        for tier in ["CRITICAL", "ELEVATED", "WATCH"]:
            rows = exp[exp["tier"] == tier]
            if rows.empty:
                continue
            if tier == "CRITICAL":
                desc = "heavily dependent on disrupted supply + almost no domestic production"
            elif tier == "ELEVATED":
                desc = "meaningful exposure to disrupted supply OR limited domestic buffer"
            else:
                desc = "some exposure but domestic production or low share provides cover"
            print()
            print("  [{}]  {}".format(tier, desc))
            print(hdr)
            print(hdr2)
            print(sep)
            for _, r in rows.iterrows():
                label  = _country_label(r["iso3"])
                dep_s  = ("{:.0f}%".format(r["import_dep_share"] * 100)
                          if pd.notna(r["import_dep_share"]) else "unknown")
                prod_s = ("{:,.0f}".format(r["crude_prod_latest"])
                          if r["crude_prod_latest"] > 0 else "near zero")
                note   = ""
                if r["iso3"] == "USA" and us_doc is not None:
                    note = "  ({:.0f} days of crude in storage)".format(us_doc)
                print("  {:<{w0}}  {:>{w1},.1f}  {:>{w2}}  {:>{w3}}  {:>{w4}}  {:>{w5}}{}".format(
                      label,
                      r["import_from_exporters"] / 1e6,
                      dep_s, prod_s,
                      r["large_airports"], r["total_airports"], note,
                      w0=W0, w1=W1, w2=W2, w3=W3, w4=W4, w5=W5))

        n_crit   = int((exp["tier"] == "CRITICAL").sum())
        n_elev   = int((exp["tier"] == "ELEVATED").sum())
        n_wtch   = int((exp["tier"] == "WATCH").sum())
        lrg_risk = int(exp[exp["tier"].isin(["CRITICAL", "ELEVATED"])]["large_airports"].sum())
        print()
        print("  Summary:")
        print("    Countries at CRITICAL risk : {}".format(n_crit))
        print("    Countries at ELEVATED risk : {}".format(n_elev))
        print("    Countries at WATCH         : {}".format(n_wtch))
        print("    Large airports in CRITICAL + ELEVATED countries: {:,}".format(lrg_risk))
        if us_doc is not None:
            print("    US crude oil stockpile covers approx. {:.0f} days at current import rates".format(
                  us_doc))
        if days_of_cover:
            print("    Analyst-provided days of fuel cover: {}".format(days_of_cover))

        exposed_iso3 = set(exp["iso3"])
        head.append("{} countries CRITICAL / {} ELEVATED".format(n_crit, n_elev))

    # -- Indo-Pacific strategic buffer ----------------------------------------
    INDOPACIFIC_ISO3 = {
        "JPN","KOR","TWN","PHL","VNM","SGP","MYS","IDN",
        "AUS","NZL","THA","IND","BGD","PAK","LKA","MMR",
        "CHN","BRN","PNG",
    }
    # Approximate days of net crude import cover.
    # IEA obligation for member countries = 90 days.
    # Sources: IEA Monthly Oil Statistics, national energy agencies (2023-2024 values).
    INDOPACIFIC_SPR_DAYS = {
        "JPN": 150,  # Japan — METI; government + industry; well above IEA obligation
        "KOR": 100,  # South Korea — KNOC managed
        "SGP":  80,  # Singapore — industry stocks; regional refining hub
        "TWN":  60,  # Taiwan — National Oil Corp estimate
        "AUS":  54,  # Australia — below IEA 90-day obligation
        "MYS":  30,  # Malaysia — Petronas managed stocks
        "NZL":  30,  # New Zealand
        "THA":  25,  # Thailand
        "IDN":  25,  # Indonesia — Pertamina managed
        "PHL":  15,  # Philippines — limited strategic storage
        "VNM":  15,  # Vietnam
        "IND":   9,  # India — very limited government reserves; mostly commercial
        "CHN":  90,  # China — SPR + commercial combined; actual figures not public
    }

    if not bilateral_exp.empty and "iso3" in bilateral_exp.columns:
        ip_exposed = exp[exp["iso3"].isin(INDOPACIFIC_ISO3)].copy()
    else:
        ip_exposed = pd.DataFrame()

    if not ip_exposed.empty:
        ip_spr = {iso3: days for iso3, days in INDOPACIFIC_SPR_DAYS.items()
                  if iso3 in ip_exposed["iso3"].values}

        # Try to supplement with live JODI closing stock data
        if "jodi_stocks" in globals() and not jodi_stocks.empty                 and "iso3" in jodi_stocks.columns and "OBS_VALUE" in jodi_stocks.columns:
            jodi_latest = (jodi_stocks[jodi_stocks["iso3"].isin(INDOPACIFIC_ISO3)]
                           .sort_values("TIME_PERIOD")
                           .groupby("iso3")["OBS_VALUE"].last())
            if not jodi_latest.empty:
                print()
                print("   Note: JODI closing stock data available for {} Indo-Pacific countries "
                      "(shown as 'thousand barrels').".format(len(jodi_latest)))

        if ip_spr:
            print()
            print("  INDO-PACIFIC STRATEGIC BUFFER")
            print("  Approximate days of crude cover — how long each country can operate")
            print("  without new imports before stocks run out.")
            print("  IEA members are obligated to hold ≥ 90 days of net import cover.")
            print()
            print("  {:<26}  {:>12}  {}".format("Country", "Days Cover", ""))
            print("  {}  {}  {}".format("-"*26, "-"*12, "-"*22))
            for iso3, days in sorted(ip_spr.items(),
                                     key=lambda x: x[1], reverse=True):
                try:
                    cname = pycountry.countries.get(alpha_3=iso3).name
                    if len(cname) > 22: cname = cname[:20] + ".."
                except Exception:
                    cname = iso3
                label   = "{} ({})".format(cname, iso3)
                iea_min = "IEA member" if iso3 in {"JPN","KOR","AUS","NZL"} else "non-IEA"
                flag    = "✓ above min" if days >= 90 else ("✗ BELOW min" if iea_min == "IEA member" else "")
                print("  {:<26}  {:>8} days  {} {}".format(
                      label, days, iea_min, flag).rstrip())
            print()
            print("  Source: IEA Monthly Oil Statistics / national energy agencies (approximate).")
            print("  Figures change monthly. Japan and South Korea have the largest buffers.")


    print("\n" + "-" * 80)
    print("OVERALL ASSESSMENT:")
    for item in head:
        print("  .  {}".format(item))
    if not head:
        print("  Insufficient data to assess.")
    print("=" * 80)


def run_top_candidate(candidates_df, date=None, days_of_cover=None):
    """Run trace_event() on the top candidate from discover_events()."""
    import datetime
    if candidates_df is None or candidates_df.empty:
        print("No candidates found. Run the Event Discovery section first.")
        return
    date     = date or datetime.date.today().isoformat()
    inv      = candidates_df[candidates_df["classification"] == "INVESTIGATE"]
    top      = inv.iloc[0] if not inv.empty else candidates_df.iloc[0]
    iso3     = top["iso3"]
    full_cp  = top.get("chokepoint") if pd.notna(top.get("chokepoint")) else None
    short_cp = PW_TO_SHORT.get(full_cp) if full_cp else None
    if not short_cp:
        for cp, exporters in CHOKEPOINT_EXPORTERS.items():
            if iso3 in exporters:
                short_cp = cp
                break
    if not short_cp:
        print("Top candidate {} does not map to a known chokepoint.".format(iso3))
        print("Add it to CHOKEPOINT_EXPORTERS to enable a full trace.")
        return
    label = _country_label(iso3)
    print("Top event candidate : {}  (news spike {:.0f}x above baseline)".format(
          label, top["spike_ratio"]))
    print("Chokepoint          : {}".format(full_cp or short_cp))
    print("Classification      : {}".format(top["classification"]))
    print()
    trace_event(short_cp, date, days_of_cover=days_of_cover)


In [20]:
import datetime as _dt

# =============================================================================
# CONTROL PANEL  --  change TRACE_TARGET, then re-run this cell
# =============================================================================
#
# HOW TO PICK AN EVENT  (choose whichever format is easiest):
#
#   By number      TRACE_TARGET = 1          # 1 = top candidate, 2 = second, etc.
#   By country     TRACE_TARGET = "Iran"     # full or partial country name
#   By code        TRACE_TARGET = "IRN"      # ISO-3 country code
#   By chokepoint  TRACE_TARGET = "Hormuz"   # chokepoint name
#   Automatic      TRACE_TARGET = "auto"     # top-ranked INVESTIGATE candidate
#   All events     TRACE_TARGET = "all"      # every INVESTIGATE candidate
#
# =============================================================================

TRACE_TARGET = "auto"

# Days of cover (how many days of fuel stockpile a country has).
# Set to a number if you know it; leave as None to use EIA data where available.
DAYS_OF_COVER = None

# =============================================================================

def _resolve_target(target, cdf):
    """
    Resolve TRACE_TARGET to a row in the candidates DataFrame.
    Accepts: int rank, ISO-3 code, country name (partial), or chokepoint name.
    Returns the matching subset of cdf, or empty DataFrame if not found.
    """
    if cdf is None or cdf.empty:
        return pd.DataFrame()

    # Rank number: 1 = top candidate
    try:
        rank = int(target)
        if 1 <= rank <= len(cdf):
            return cdf.iloc[[rank - 1]]
        print("  Rank {} is out of range. There are {} candidates.".format(rank, len(cdf)))
        return pd.DataFrame()
    except (ValueError, TypeError):
        pass

    t = str(target).strip()

    # Exact ISO-3 match
    iso_match = cdf[cdf["iso3"] == t.upper()]
    if not iso_match.empty:
        return iso_match

    # Chokepoint name match (partial, case-insensitive)
    if "chokepoint" in cdf.columns:
        cp_match = cdf[cdf["chokepoint"].str.contains(t, case=False, na=False)]
        if not cp_match.empty:
            return cp_match

    # Country name match via pycountry (partial)
    t_low = t.lower()
    matched_iso3 = []
    for _c in pycountry.countries:
        if t_low in _c.name.lower():
            matched_iso3.append(_c.alpha_3)
    if matched_iso3:
        name_match = cdf[cdf["iso3"].isin(matched_iso3)]
        if not name_match.empty:
            return name_match

    return pd.DataFrame()


if "candidates" not in dir() or candidates.empty:
    print("No candidates found. Run the Event Discovery cell first.")
else:
    # -- Numbered candidate menu ----------------------------------------------
    print("=" * 72)
    print("CANDIDATE EVENTS -- pick one to trace")
    print("=" * 72)
    print()
    print("  {:<3}  {:<24}  {:<22}  {:>7}  {}".format(
          "#", "Country", "Chokepoint", "Spike", "Status"))
    print("  {}  {}  {}  {}  {}".format("-"*3, "-"*24, "-"*22, "-"*7, "-"*12))

    for i, (_, row) in enumerate(candidates.iterrows(), start=1):
        try:
            cname = pycountry.countries.get(alpha_3=row["iso3"]).name
            if len(cname) > 20:
                cname = cname[:18] + ".."
        except Exception:
            cname = row["iso3"]
        label = "{} ({})".format(cname, row["iso3"])
        cp    = str(row["chokepoint"]) if pd.notna(row.get("chokepoint")) else "--"

        # Mark current selection
        marker = " <-- current" if str(TRACE_TARGET) == str(i) or (
            isinstance(TRACE_TARGET, str) and (
                TRACE_TARGET.upper() == row["iso3"] or
                TRACE_TARGET.lower() in str(row.get("chokepoint", "")).lower() or
                TRACE_TARGET.lower() in cname.lower()
            )
        ) else ""
        if TRACE_TARGET == "auto" and i == 1:
            marker = " <-- current (auto)"

        print("  {:<3}  {:<24}  {:<22}  {:>6.0f}x  {}{}".format(
              i, label, cp, row["spike_ratio"], row["classification"], marker))

    print()
    print("  Change TRACE_TARGET above to pick a different event:")
    print("    TRACE_TARGET = 1          # by number")
    print("    TRACE_TARGET = \"IRN\"      # by ISO-3 code")
    print("    TRACE_TARGET = \"Iran\"     # by country name")
    print("    TRACE_TARGET = \"Hormuz\"   # by chokepoint")
    print("    TRACE_TARGET = \"all\"      # run every INVESTIGATE candidate")
    print()
    print("  Current: TRACE_TARGET = {}  |  DAYS_OF_COVER = {}".format(
          repr(TRACE_TARGET), DAYS_OF_COVER))
    print("=" * 72)
    print()

    # -- Resolve and run ------------------------------------------------------
    if TRACE_TARGET == "auto":
        run_top_candidate(candidates, days_of_cover=DAYS_OF_COVER)

    elif TRACE_TARGET == "all":
        inv = candidates[candidates["classification"] == "INVESTIGATE"]
        targets = inv if not inv.empty else candidates
        for _, row in targets.iterrows():
            run_top_candidate(
                candidates[candidates["iso3"] == row["iso3"]],
                days_of_cover=DAYS_OF_COVER,
            )
            print()

    else:
        subset = _resolve_target(TRACE_TARGET, candidates)
        if subset.empty:
            print('  Could not find "{}". Valid options:'.format(TRACE_TARGET))
            for i, (_, row) in enumerate(candidates.iterrows(), start=1):
                try:
                    cname = pycountry.countries.get(alpha_3=row["iso3"]).name
                except Exception:
                    cname = row["iso3"]
                cp = str(row["chokepoint"]) if pd.notna(row.get("chokepoint")) else "--"
                print("    {}  {} ({})  --  {}".format(i, cname, row["iso3"], cp))
        else:
            run_top_candidate(subset, days_of_cover=DAYS_OF_COVER)


CANDIDATE EVENTS -- pick one to trace

  #    Country                   Chokepoint                Spike  Status
  ---  ------------------------  ----------------------  -------  ------------
  1    Iran, Islamic Repu.. (IRN)  Strait of Hormuz           280x  INVESTIGATE <-- current (auto)
  2    United States (USA)       --                         154x  INVESTIGATE
  3    Yemen (YEM)               Bab el-Mandeb Strait        42x  INVESTIGATE
  4    Saudi Arabia (SAU)        --                          18x  INVESTIGATE
  5    India (IND)               --                          10x  INVESTIGATE
  6    Israel (ISR)              Suez Canal                   8x  INVESTIGATE
  7    Venezuela, Bolivar.. (VEN)  --                           8x  INVESTIGATE
  8    Qatar (QAT)               --                           6x  INVESTIGATE

  Change TRACE_TARGET above to pick a different event:
    TRACE_TARGET = 1          # by number
    TRACE_TARGET = "IRN"      # by ISO-3 code
    TRACE_TARGET =


EVENT START ESTIMATE
   The tool looks for two signals to pin down when this disruption began.

   News spike first detected   : Jun 23, 2026  (GDELT)
   This is when global news coverage of the affected region first
   crossed double its normal level — the market was paying attention.

   Tanker volume drop detected : Apr 26, 2026  (PortWatch)
   This is when ship traffic first dropped >20% below its prior 30-day
   average — ships physically stopped moving through the chokepoint.

   Baseline anchor : Apr 26, 2026  --  PortWatch — first confirmed tanker volume drop
   All before/after comparisons below are anchored to this date.
------------------------------------------------------------------------

(1) NEWS SIGNAL   (how much is this event dominating media coverage?)


   Coverage period : Jun 23, 2026 to Jul 22, 2026  (29 day span)
   First appeared  : Jun 23, 2026
   Peak coverage   : Jul 22, 2026  (82 articles that day)
   Trend           : still escalating
   Daily article volume (mentions of exporter countries):
     Jun 23  █████████████████████████       69
     Jul 22  ██████████████████████████████  82
   Spike ratio     : unreliable (no historical baseline -- API cap hit)
   Note: news signal shows WHERE to look -- it does not confirm supply was disrupted.

(2) PHYSICAL DISRUPTION CHECK   (did ships actually stop moving?)
   Source: IMF PortWatch -- counts tanker ships passing through the chokepoint each day.
   Chokepoint: Strait of Hormuz
   Tanker ships in the last 7 days   : 3.5 per day
   Tanker ships before the event     : 7.4 per day  (average from 14 to 45 days ago)
   Current traffic as % of normal    : 47%
   Result: CONFIRMED PHYSICAL DISRUPTION -- tanker traffic has collapsed

   US crude oil in storage: 409,665 thousand barrels

## Extensions (add when ready)

- **GDELT (events):** `pip install gdeltdoc` for light pulls, or query the
  `gdelt-bq.gdeltv2.events` public dataset in Google BigQuery with SQL — filter to a
  country/date/event-type and pull just that slice.
- **ACLED (verified events):** REST API with a key (free, non-commercial).
- **IMF PortWatch (chokepoint transits + Economy Monitor):** its datasets live on an
  ArcGIS Hub — each has a CSV download and a queryable REST endpoint (`.../query?where=...&f=json`).
- **OPEC MOMR (tables):** parse the monthly PDF with `pdfplumber` / `camelot` for the
  prices, demand, supply, stocks, and days-of-cover tables.
- **GPR Index (risk premium):** `pd.read_excel` from the Caldara–Iacoviello file.

**Refresh:** wrap the loaders in a scheduled run (cron, or a `schedule` loop) so the cache
refreshes daily; the `cached()` helper already skips re-fetching fresh data.

**Then:** `trace_event()` is where the data becomes a story — extend it to pull live prices
(§5), jet-fuel (IATA), and PortWatch transits so each of the five cascade layers is filled
from real data.
